# Pipeline de datos COES — Centrales solares del SEIN
### Gridvia Analytics · Pronóstico day-ahead AutoML

Notebook único: **descarga → consolidación → calidad → plantilla de limpieza**.

| Sección | Qué hace | Modo |
|---|---|---|
| 0 | Instalación de dependencias (correr una vez) | Auto |
| 1 | Configuración | — |
| 2 | **Descarga COES** (CSV mensual) — sesión activa | **MANUAL** |
| 3 | Consolidación → 16 centrales, formato largo | Auto |
| 4 | Reporte de calidad por central | Auto |
| 5 | Plantilla de limpieza | Manual |

**Decisiones cerradas** (no cambiar sin razón):
- Agrupar por **etiqueta interna** (post ` -`), NO por empresa → resuelve cambios de dueño (Enel→Orygen para RUBÍ y CLEMESÍ).
- INTIPAMPA y EXPANSION_INTIPAMPA = `item_id` **separados** → evita escalón artificial de capacidad.
- Normalización agresiva de etiquetas (tildes, espacios, encoding) → robusto a inconsistencias del COES.

## 0. Instalación de dependencias (una sola vez)

In [ ]:
import sys, subprocess
for pkg in ["pyarrow", "requests", "pandas"]:
    try: __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("OK")

## 1. Configuración

In [ ]:
import os, glob, time, calendar, re, unicodedata
import importlib
import requests
import pandas as pd
import numpy as np

ANIO_INI, MES_INI = 2018, 1
ANIO_FIN, MES_FIN = 2026, 5
RAW_DIR = "data/raw/coes_csv"
os.makedirs(RAW_DIR, exist_ok=True);
BASE = "https://www.coes.org.pe/Portal/mediciones/medidoresgeneracion"
EMPRESAS = ("14260,69,15214,11772,12439,11777,10481,12056,12362,13196,12896,10628,15707,"
"10420,12708,13165,10901,15571,12584,11095,17,11153,58,19,10684,30,27,40,11228,23,24,2,"
"11389,11129,14173,11412,206,11544,11058,12097,10552,11429,12884,15624,15392,10725,11527,"
"10647,14973,11981,11840,11841,12974,10974,11258,11644,11444,11940,12364,11149,11528,12634,"
"12624,13726,12479,14807,11185,13,10916,13965,67,108,11217,12480,15259,11102,13120,11218,"
"11063,149,11323,13966,15009,61,10913,10587,8,138,12758,11103,15080,10,15683,14342,13984,"
"167,10422,180,11146,15014,4,5,9,76,11563,10636,11509,12096,11395,13783,18,48,10582,11064,"
"47,11100,10755,12190,11053,10984,11101,11486,6,11567,10767,11894")
print("Config OK:", (ANIO_INI, MES_INI), "->", (ANIO_FIN, MES_FIN), "| CSV (tipo=3)")

## 2. Descarga COES  ⚠️ MANUAL

Flujo HTTP confirmado (3 pasos, misma sesión):
`POST /validarexportacion` → `POST /exportar` → `GET /descargar?tipo=3`.

La `Session()` captura sola la cookie ASP.NET que emite el servidor en el primer GET.
Si necesitas pegar cookie manual, ponla en `cookie.txt`.

In [ ]:
def make_session():
    s = requests.Session()
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Origin": "https://www.coes.org.pe", "Referer": BASE,
        "X-Requested-With": "XMLHttpRequest",
    })
    if os.path.exists("cookie.txt"):
        ck = open("cookie.txt", encoding="utf-8").read().strip()
        if ck: s.headers["Cookie"] = ck
    s.get(BASE, timeout=60)
    return s

def payload(fi, ff):
    return {"fechaInicial": fi, "fechaFinal": ff, "empresas": EMPRESAS,
            "tiposGeneracion": "3", "central": "1", "parametros": "1", "tipo": "3"}

def es_csv(r):
    ct = r.headers.get("Content-Type", "").lower()
    return r.status_code == 200 and len(r.content) > 2000 and ("csv" in ct or "text" in ct or "octet" in ct)

def bajar_mes(s, fi, ff):
    form = {"Content-Type": "application/x-www-form-urlencoded; charset=UTF-8"}
    s.post(BASE + "/validarexportacion", headers=form, data=payload(fi, ff), timeout=120)
    s.post(BASE + "/exportar",           headers=form, data=payload(fi, ff), timeout=120)
    return s.get(BASE + "/descargar", params={"tipo": "3"}, timeout=120)

print("Funciones de descarga listas.")

### 2a. Prueba de UN mes (córrela primero)

In [ ]:
s = make_session()
r = bajar_mes(s, "01/01/2018", "30/01/2018")
print("status:", r.status_code, "| type:", r.headers.get("Content-Type","")[:40], "| bytes:", len(r.content))
print("primeras líneas:")
print(r.content[:300].decode("utf-8", errors="replace"))

### 2b. Descarga masiva (reanudable)

In [ ]:
def descargar_todo():
    s = make_session()
    y, m = ANIO_INI, MES_INI
    while (y, m) <= (ANIO_FIN, MES_FIN):
        out = f"{RAW_DIR}/solar_{y}_{m:02d}.csv"
        if os.path.exists(out) and os.path.getsize(out) > 2000:
            print(f"  {y}-{m:02d}  existe, salto")
        else:
            ult = calendar.monthrange(y, m)[1]
            try:
                r = bajar_mes(s, f"01/{m:02d}/{y}", f"{ult:02d}/{m:02d}/{y}")
                if es_csv(r):
                    open(out, "wb").write(r.content)
                    print(f"  {y}-{m:02d}  OK ({len(r.content)//1024} KB)")
                else:
                    print(f"  {y}-{m:02d}  FALLO ({r.status_code}) -> renuevo sesión")
                    s = make_session()
            except Exception as e:
                print(f"  {y}-{m:02d}  ERROR {e}")
            time.sleep(2.5)
        m += 1
        if m > 12: m = 1; y += 1
    print("Listo:", RAW_DIR)

descargar_todo()


## 3. Consolidación → 16 centrales, formato largo

**Normalización agresiva** de etiquetas (mayúsculas, sin tildes, espacios → `_`)
para que `SUNNY EXPANSIÓN-BL1`, `SUNNY EXPANSION-BL1`, etc. caigan todas en la
misma clave canónica. Validado contra cabeceras de 2018, 2020, 2022, 2024 y 2026.

In [ ]:
# === Sección 3: Consolidación → un solo DataFrame en formato largo ===
import os, glob, re, unicodedata
import pandas as pd
import numpy as np

MAPEO = {
    "RUBI_CIRCUITO_1-5":    "RUBI",
    "RUBI_CIRCUITO_6-10":   "RUBI",
    "GTSMAJES":             "MAJES",
    "MOQUEGUA_SOLAR":       "MOQUEGUA_FV",
    "PANAM_SOLAR":          "PANAMERICANA",
    "GTSREPAR":             "REPARTICION",
    "TACNA_SOLAR":          "TACNA",
    "YARUCAYA":             "YARUCAYA",
    "INTIPAMPA_SOLAR":      "INTIPAMPA",
    "EXPANSION_INTIPAMPA":  "EXPANSION_INTIPAMPA",
    "COENERGY":             "COENERGY",
    "MATARANI":             "MATARANI",
    "SAN_MARTIN_SOLAR-BL1": "SAN_MARTIN",
    "SAN_MARTIN_SOLAR-BL2": "SAN_MARTIN",
    "SUNNY-BL1":            "SUNNY",
    "SUNNY-BL2":            "SUNNY",
    "SUNNY_EXPANSION-BL1":  "SUNNY_EXPANSION",
    "SUNNY_EXPANSION-BL2":  "SUNNY_EXPANSION",
    "CARHUAQUERO_CS":       "CARHUAQUERO",
    "CLEMESI":              "CLEMESI",
}

def normalizar_etiqueta(col):
    p = col.split(" -", 1)[-1].strip().upper()
    p = "".join(c for c in unicodedata.normalize("NFD", p)
                if unicodedata.category(c) != "Mn")
    p = re.sub(r"\s+", "_", p)
    p = re.sub(r"_+", "_", p)
    return p

def leer_csv_mes(path):
    df = None
    for enc in ("utf-8", "latin-1"):
        try:
            df = pd.read_csv(path, encoding=enc, skipinitialspace=True); break
        except Exception:
            continue
    if df is None: raise IOError(path)
    df.columns = [c.strip() for c in df.columns]
    tcol = df.columns[0]
    df[tcol] = pd.to_datetime(df[tcol], dayfirst=True, errors="coerce")
    df = df[df[tcol].notna()].set_index(tcol); df.index.name = "timestamp"
    salida = {}
    for col in df.columns:
        c = MAPEO.get(normalizar_etiqueta(col))
        if c is None: continue
        serie = pd.to_numeric(df[col], errors="coerce")
        salida[c] = (salida[c] + serie) if c in salida else serie
    return pd.DataFrame(salida)

archivos = sorted(glob.glob(f"{RAW_DIR}/*.csv"))
print(f"Leyendo {len(archivos)} archivos...")
_partes = [leer_csv_mes(a) for a in archivos]
_full = pd.concat(_partes).sort_index()
_full = _full[~_full.index.duplicated(keep="first")]
_ancho_tmp = _full.resample("1h", label="right", closed="right").mean()
#_ancho_tmp = _full[_full.index.minute == 0]   # solo timestamps en hora en punto

# === ÚNICO df persistente: formato largo ===
largo = (_ancho_tmp.reset_index()
                   .melt(id_vars="timestamp", var_name="item_id", value_name="potencia_mw")
                   .dropna(subset=["potencia_mw"])
                   .sort_values(["item_id", "timestamp"])
                   .reset_index(drop=True))

del _partes, _full, _ancho_tmp   # liberar memoria; queda solo `largo`

print(f"\n`largo`: DataFrame {largo.shape}")
print(f"Centrales (item_id): {largo.item_id.nunique()}")
print(f"Rango: {largo.timestamp.min()} -> {largo.timestamp.max()}")
print(largo.head())

## 4. Reporte de calidad por central
Modifica `largo` directamente. No toca ceros internos ni recortes — solo descarta.

In [ ]:
# plantas con historia operativa insuficiente para cubrir ciclo estacional completo.

UMBRAL_INCLUSION_MESES = 18   # Chen 2026, Despotovic 2024, Ray 2024
UMBRAL_GEN = 0.05             # MW, para determinar inicio operativo real

# Calcular meses operativos por planta (basado en primera/última generación >umbral)
meses_operativos = {}
for central, g in largo.groupby("item_id"):
    s_real = g[g.potencia_mw > UMBRAL_GEN]
    if len(s_real) == 0:
        meses_operativos[central] = 0
        continue
    ini = s_real.timestamp.min()
    fin = s_real.timestamp.max()
    horas = (fin - ini).total_seconds() / 3600 + 1
    meses_operativos[central] = round(horas / 730, 1)

# Clasificar
INCLUIDAS = [c for c, m in meses_operativos.items() if m >= UMBRAL_INCLUSION_MESES]
EXCLUIDAS = [c for c, m in meses_operativos.items() if m <  UMBRAL_INCLUSION_MESES]

# Filtrar `largo` directamente (única estructura persistente)
n_antes = len(largo)
largo = largo[largo.item_id.isin(INCLUIDAS)].reset_index(drop=True)
n_despues = len(largo)

# Resumen
print(f"Umbral de inclusión: ≥ {UMBRAL_INCLUSION_MESES} meses operativos (ciclo estacional completo)\n")

print(f"INCLUIDAS ({len(INCLUIDAS)} plantas):")
for c in sorted(INCLUIDAS, key=lambda x: -meses_operativos[x]):
    print(f"  - {c:22} ({meses_operativos[c]:5.1f} meses)")

print(f"\nEXCLUIDAS ({len(EXCLUIDAS)} plantas, <{UMBRAL_INCLUSION_MESES}m):")
for c in sorted(EXCLUIDAS, key=lambda x: -meses_operativos[x]):
    print(f"  - {c:22} ({meses_operativos[c]:5.1f} meses)")

print(f"\n`largo`: {n_antes:,} → {n_despues:,} filas ({n_antes - n_despues:,} eliminadas)")
print(f"Plantas en `largo`: {largo.item_id.nunique()}")

## 4. Reporte de calidad por central

Detecta (no borra): primer dato real, meses de serie, % de huecos, ceros diurnos
sospechosos, y marca centrales cuyo primer mes parece comisionamiento.

In [ ]:
# === Sección 4b corregida: Tabla insumo meteo con fecha exacta del primer registro ===
# inicio_csv = primer timestamp de la planta en el CSV consolidado (puede ser cero)
# inicio_real = primer timestamp con generación > umbral (lo que tiene sentido para Solcast)

import requests
import pandas as pd

UMBRAL_GEN = 0.05

NOMBRE_GIS = {
    "RUBI":         "RUBI",
    "PANAMERICANA": "PANAMERICANA SOLAR",
    "TACNA":        "TACNA SOLAR",
    "MAJES":        "MAJES",
    "MOQUEGUA_FV":  "MOQUEGUA SOLAR",
    "REPARTICION":  "REPARTICION",
    "INTIPAMPA":    "INTIPAMPA",
    "YARUCAYA":     "YARUCAYA",
    "CLEMESI":      "CLEMESÍ",
    "CARHUAQUERO":  "CARHUAQUERO",
    "MATARANI":     "MATARANI",
    "SAN_MARTIN":   "SAN MARTIN SOLAR",
}

URL_GIS = ("https://gisem.osinergmin.gob.pe/serverosih/rest/services/"
           "Electricidad/ELECTRICIDAD/MapServer/11/query")
params = {
    "where": "TIPO_CENTRAL='CENTRAL SOLAR'",
    "outFields": "EMPRESA,NOMBRE,POT_INST,LATITUD,LONGITUD,PUEST_OPER",
    "returnGeometry": "false",
    "f": "json",
}
r = requests.get(URL_GIS, params=params, timeout=60)
gis = {feat["attributes"]["NOMBRE"]: feat["attributes"] for feat in r.json()["features"]}

filas = []
for item_id in sorted(largo.item_id.unique()):
    g = largo[largo.item_id == item_id].sort_values("timestamp")

    # Inicio exacto del CSV: primer timestamp donde la planta aparece (incluye ceros)
    inicio_csv = g["timestamp"].min()
    fin_csv = g["timestamp"].max()

    # Inicio real de generación: primer timestamp con potencia > umbral
    g_real = g[g.potencia_mw > UMBRAL_GEN]
    inicio_real = g_real["timestamp"].min() if len(g_real) else pd.NaT
    fin_real    = g_real["timestamp"].max() if len(g_real) else pd.NaT

    horas_csv  = int((fin_csv  - inicio_csv ).total_seconds() / 3600) + 1
    horas_real = int((fin_real - inicio_real).total_seconds() / 3600) + 1 if pd.notna(inicio_real) else 0

    nombre_gis = NOMBRE_GIS.get(item_id)
    if nombre_gis is None or nombre_gis not in gis:
        print(f"AVISO: {item_id} sin coordenada en GIS"); continue
    info = gis[nombre_gis]

    filas.append({
        "item_id":      item_id,
        "lat":          info["LATITUD"],
        "lon":          info["LONGITUD"],
        "pot_inst_mw":  info["POT_INST"],
        "inicio_csv":   inicio_csv,    # ← fecha y hora exacta del primer registro
        "fin_csv":      fin_csv,       # ← fecha y hora exacta del último registro
        "horas_csv":    horas_csv,
        "inicio_real":  inicio_real,   # ← primer timestamp con generación real
        "fin_real":     fin_real,
        "horas_real":   horas_real,
    })

df_meteo_input = pd.DataFrame(filas).sort_values("item_id").reset_index(drop=True)

from IPython.display import display
display(df_meteo_input)
print(f"\nTotal plantas: {len(df_meteo_input)}")
print(f"Total horas-CSV  (suma): {df_meteo_input.horas_csv.sum():,}")
print(f"Total horas-real (suma): {df_meteo_input.horas_real.sum():,}")
print(f"Diferencia (horas pre/post-operativas): {df_meteo_input.horas_csv.sum() - df_meteo_input.horas_real.sum():,}")

In [ ]:
import requests, csv

URL = "https://gisem.osinergmin.gob.pe/serverosih/rest/services/Electricidad/ELECTRICIDAD/MapServer/11/query"
params = {
    "where": "TIPO_CENTRAL='CENTRAL SOLAR'",
    "outFields": "NOMBRE,LATITUD,LONGITUD,POT_INST",
    "returnGeometry": "false",
    "f": "json",
}
r = requests.get(URL, params=params, timeout=60)
for f in r.json()["features"]:
    a = f["attributes"]
    print(f"{a['NOMBRE']},{a['LATITUD']},{a['LONGITUD']}")

In [ ]:
# === Sección 5 — parte 2 (definitiva): Descarga ERA5-Land timeseries por planta ===
# Endpoint: reanalysis-era5-land-timeseries (Zarr cloud-optimized).
# Reanudable: salta plantas ya descargadas.
# Rutas absolutas relativas al PROJECT_DIR (no depende del cwd del kernel).

import cdsapi
import os
import pandas as pd

# Ruta absoluta del proyecto y subdirectorios
PROJECT_DIR   = os.getcwd()
CDSAPIRC_PATH = os.path.join(PROJECT_DIR, ".cdsapirc")
ERA5_DIR      = os.path.join(PROJECT_DIR, "data", "raw", "era5_land")
os.makedirs(ERA5_DIR, exist_ok=True)

print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"ERA5_DIR:    {ERA5_DIR}")

# Cargar credenciales
def cargar_credenciales(path):
    creds = {}
    with open(path) as f:
        for linea in f:
            if ":" not in linea: continue
            k, _, v = linea.partition(":")
            creds[k.strip()] = v.strip()
    return creds.get("url"), creds.get("key")

CDS_URL, CDS_KEY = cargar_credenciales(CDSAPIRC_PATH)
if not (CDS_URL and CDS_KEY):
    raise RuntimeError(f"No se pudo leer credenciales de {CDSAPIRC_PATH}")

# Variables (7, sin forecast_albedo que no está disponible en timeseries)
VARIABLES = [
    "surface_solar_radiation_downwards",   # GHI, de-accumulated
    "2m_temperature",
    "2m_dewpoint_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "surface_pressure",
    "total_precipitation",                  # de-accumulated
]

def descargar_planta_timeseries(client, item_id, lat, lon, inicio, fin):
    """Descarga ERA5-Land timeseries para una planta. Ruta absoluta garantizada."""
    out_path = os.path.join(ERA5_DIR, f"{item_id}.nc")   # ← absoluta
    if os.path.exists(out_path) and os.path.getsize(out_path) > 10_000:
        mb = os.path.getsize(out_path) // 1024 // 1024
        print(f"  {item_id:14}  ya existe ({mb} MB), salto")
        return

    fecha_rango = f"{inicio.strftime('%Y-%m-%d')}/{fin.strftime('%Y-%m-%d')}"
    print(f"  {item_id:14}  pidiendo ({fecha_rango})...")

    try:
        client.retrieve(
            "reanalysis-era5-land-timeseries",
            {
                "variable": VARIABLES,
                "location": {"longitude": lon, "latitude": lat},
                "date":     [fecha_rango],
                "data_format": "netcdf",
            },
            out_path,
        )
        mb = os.path.getsize(out_path) // 1024 // 1024
        print(f"  {item_id:14}  OK ({mb} MB) → {out_path}")
    except Exception as e:
        print(f"  {item_id:14}  ERROR: {e}")
        if os.path.exists(out_path):
            os.remove(out_path)

# Cliente CDS
client = cdsapi.Client(url=CDS_URL, key=CDS_KEY)

# Loop sobre las 11 plantas
for _, fila in df_meteo_input.iterrows():
    descargar_planta_timeseries(
        client,
        item_id = fila["item_id"],
        lat     = fila["lat"],
        lon     = fila["lon"],
        inicio  = pd.Timestamp(fila["inicio_csv"]),
        fin     = pd.Timestamp(fila["fin_csv"]),
    )

# Verificación final: listar lo descargado
print(f"\n=== Archivos en {ERA5_DIR} ===")
for f in sorted(os.listdir(ERA5_DIR)):
    path = os.path.join(ERA5_DIR, f)
    mb = os.path.getsize(path) // 1024 // 1024
    print(f"  {f:25} {mb:>5} MB")

In [ ]:
# === Descarga adicional: Total Cloud Cover (TCC) ===

ERA5_TCC_DIR = os.path.join(PROJECT_DIR, "data", "raw", "era5_tcc")
os.makedirs(ERA5_TCC_DIR, exist_ok=True)

def descargar_tcc(client, item_id, lat, lon, inicio, fin):

    out_path = os.path.join(
        ERA5_TCC_DIR,
        f"{item_id}_tcc.nc"
    )

    if os.path.exists(out_path) and os.path.getsize(out_path) > 10_000:
        print(f"  {item_id:14} ya existe, salto")
        return

    fecha_rango = (
        f"{inicio.strftime('%Y-%m-%d')}/"
        f"{fin.strftime('%Y-%m-%d')}"
    )

    client.retrieve(
        "reanalysis-era5-single-levels-timeseries",
        {
            "variable": ["total_cloud_cover"],
            "location": {
                "longitude": lon,
                "latitude": lat,
            },
            "date": [fecha_rango],
            "data_format": "netcdf",
        },
        out_path,
    )

for _, fila in df_meteo_input.iterrows():

    descargar_tcc(
        client,
        item_id=fila["item_id"],
        lat=fila["lat"],
        lon=fila["lon"],
        inicio=pd.Timestamp(fila["inicio_csv"]),
        fin=pd.Timestamp(fila["fin_csv"]),
    )

In [ ]:
import zipfile
import os

ERA5_DIR = os.path.join(os.getcwd(), "data", "raw", "era5_land")
ejemplo = os.path.join(ERA5_DIR, "RUBI.nc")

with zipfile.ZipFile(ejemplo, "r") as zf:
    print(f"Contenido del ZIP ({len(zf.namelist())} entradas):")
    for nombre in zf.namelist()[:20]:
        info = zf.getinfo(nombre)
        print(f"  {nombre:60} {info.file_size:>10,} bytes")
    if len(zf.namelist()) > 20:
        print(f"  ... ({len(zf.namelist()) - 20} más)")

In [ ]:
# === Sección 6 - parte 1: Crear `largo_meteo` desde los 11 ZIPs ===

import zipfile
import io
import os
import numpy as np
import pandas as pd
import xarray as xr

ERA5_DIR = os.path.join(os.getcwd(), "data", "raw", "era5_land")

def leer_era5_planta(item_id):
    zip_path = os.path.join(ERA5_DIR, f"{item_id}.nc")
    if not os.path.exists(zip_path):
        print(f"  {item_id:14}  archivo no encontrado, salto")
        return None
    datasets = []
    with zipfile.ZipFile(zip_path, "r") as zf:
        for nombre in zf.namelist():
            raw = zf.read(nombre)
            datasets.append(xr.open_dataset(io.BytesIO(raw), engine="h5netcdf"))
    ds = xr.merge(datasets)
    df = ds.to_dataframe().reset_index()
    df = df.rename(columns={"valid_time": "timestamp"})
    for col in ["latitude", "longitude", "number", "expver"]:
        if col in df.columns:
            df = df.drop(columns=col)
    df["item_id"] = item_id
    return df.sort_values("timestamp").reset_index(drop=True)

# Loop sobre las 11 plantas de df_meteo_input
piezas = []
for item_id in df_meteo_input["item_id"]:
    print(f"  {item_id:14}  parseando...")
    df_p = leer_era5_planta(item_id)
    if df_p is not None:
        piezas.append(df_p)
        print(f"  {item_id:14}  OK ({len(df_p):,} filas)")

largo_meteo = pd.concat(piezas, ignore_index=True)

# Derivar variables físicas
largo_meteo["ghi"]        = largo_meteo["ssrd"] / 3600.0
largo_meteo["t_air"]      = largo_meteo["t2m"] - 273.15
largo_meteo["dewpoint"]   = largo_meteo["d2m"] - 273.15
a, b = 17.625, 243.04
Td, T = largo_meteo["dewpoint"], largo_meteo["t_air"]
largo_meteo["hr"]         = 100.0 * np.exp((a*Td)/(b+Td)) / np.exp((a*T)/(b+T))
largo_meteo["wind_speed"] = np.sqrt(largo_meteo["u10"]**2 + largo_meteo["v10"]**2)
largo_meteo["pressure"]   = largo_meteo["sp"] / 100.0
largo_meteo["precip_mm"]  = largo_meteo["tp"] * 1000.0

# Quedarse solo con derivadas (eliminar crudas)
largo_meteo = largo_meteo[[
    "timestamp", "item_id",
    "ghi", "t_air", "hr", "wind_speed", "pressure", "precip_mm",
]]

# Corrección UTC → hora local Perú
largo_meteo["timestamp"] = largo_meteo["timestamp"] - pd.Timedelta(hours=5)

# Merge con potencia
largo_unificado = pd.merge(
    largo, largo_meteo,
    on=["timestamp", "item_id"],
    how="inner",
).sort_values(["item_id", "timestamp"]).reset_index(drop=True)

print(f"\n`largo_meteo`:     {largo_meteo.shape}")
print(f"`largo_unificado`: {largo_unificado.shape}")
print(f"\n=== RUBI 2023-06-15 (24h) ===")
muestra = largo_unificado[
    (largo_unificado.item_id == "RUBI") &
    (largo_unificado.timestamp >= "2023-06-15") &
    (largo_unificado.timestamp < "2023-06-16")
]
print(muestra.to_string(index=False))

In [ ]:
# === Inspección visual de alineación en 3 escenarios ===

def imprimir_dia(item_id, fecha, descripcion):
    print(f"\n{'='*70}")
    print(f"  {item_id}  {fecha}  —  {descripcion}")
    print('='*70)
    m = largo_unificado[
        (largo_unificado.item_id == item_id) &
        (largo_unificado.timestamp >= fecha) &
        (largo_unificado.timestamp < pd.Timestamp(fecha) + pd.Timedelta(days=1))
    ][["timestamp", "potencia_mw", "ghi", "t_air", "hr"]]
    print(m.to_string(index=False))

# 1) Día de verano austral (enero), sol alto
imprimir_dia("RUBI", "2023-01-15", "Verano austral, sol alto en Moquegua")

# 2) Una planta nueva (chica)
imprimir_dia("CARHUAQUERO", "2024-08-15", "Planta chica de 0.6 MW, en Cajamarca norte")

# 3) Otra planta para variar latitud y empresa
imprimir_dia("YARUCAYA", "2022-11-20", "Lima centro-norte, primavera austral")

In [ ]:
# === Sección 7: Limpieza determinista P vs GHI ===
# Detecta dos condiciones físicamente inverosímiles y marca potencia como NaN:
#   A) Parada operativa: pot < 1% nominal Y GHI >= 200 W/m²
#   B) Generación sin sol: pot >= 1% nominal Y GHI == 0 (inverosímil físico)
#
# Las filas NO se eliminan; solo `potencia_mw` pasa a NaN para que AutoGluon
# trate esos timestamps como faltantes durante el entrenamiento.

UMBRAL_GHI_ALTO = 200.0       # W/m², luz solar inequívoca
UMBRAL_GHI_CERO = 0.001       # W/m², numéricamente cero
PCT_NOMINAL_MIN = 0.01        # 1% de capacidad nominal

# Capacidad nominal por planta (de df_meteo_input)
nominal = df_meteo_input.set_index("item_id")["pot_inst_mw"].to_dict()

# Calcular el umbral relativo por fila (1% de su nominal)
largo_unificado["umbral_pot"] = largo_unificado["item_id"].map(nominal) * PCT_NOMINAL_MIN

# Detección A: Parada operativa (potencia ~0 con sol alto)
mask_parada = (
    (largo_unificado["potencia_mw"] < largo_unificado["umbral_pot"]) &
    (largo_unificado["ghi"] >= UMBRAL_GHI_ALTO)
)

# Detección B: Generación sin sol (físicamente imposible)
mask_inverosimil = (
    (largo_unificado["potencia_mw"] >= largo_unificado["umbral_pot"]) &
    (largo_unificado["ghi"] < UMBRAL_GHI_CERO)
)

# Total a marcar
mask_total = mask_parada | mask_inverosimil

print(f"=== Detección de inverosimilitudes ===\n")
print(f"Total filas: {len(largo_unificado):,}")
print(f"\nA) Paradas operativas (pot<1%nom & GHI>={UMBRAL_GHI_ALTO}):  {mask_parada.sum():,} filas")
print(f"B) Generación sin sol (pot>=1%nom & GHI~0):                {mask_inverosimil.sum():,} filas")
print(f"TOTAL a marcar como NaN:                                   {mask_total.sum():,} ({100*mask_total.sum()/len(largo_unificado):.2f}%)")

# Desglose por planta
print(f"\n=== Desglose por planta ===")
desglose = pd.DataFrame({
    "paradas":      largo_unificado[mask_parada].groupby("item_id").size(),
    "inverosimil":  largo_unificado[mask_inverosimil].groupby("item_id").size(),
    "total_filas":  largo_unificado.groupby("item_id").size(),
}).fillna(0).astype(int)
desglose["pct_marcado"] = (100 * (desglose["paradas"] + desglose["inverosimil"]) / desglose["total_filas"]).round(2)
print(desglose.to_string())

# Aplicar: reemplazar potencia con NaN
largo_unificado.loc[mask_total, "potencia_mw"] = np.nan

# Limpiar columna auxiliar
largo_unificado = largo_unificado.drop(columns=["umbral_pot"])

print(f"\n=== `largo_unificado` post-limpieza ===")
print(f"Shape: {largo_unificado.shape}")
print(f"Filas con potencia_mw NaN: {largo_unificado.potencia_mw.isna().sum():,}")
print(f"Filas con potencia_mw válida: {largo_unificado.potencia_mw.notna().sum():,}")

In [ ]:
# === Sección 7 (corregida): Detección de paradas DE DÍA COMPLETO ===
# Si en un día la planta tuvo >=80% de horas con sol alto a potencia ~0,
# marcar el día completo como NaN (toda la jornada, no solo las horas diurnas).

UMBRAL_GHI_ALTO = 200.0
PCT_NOMINAL_MIN = 0.01
PCT_HORAS_MIN   = 0.80   # 80% de las horas con sol alto deben estar a ~0

# Reset: traer la potencia limpia desde `largo` (sin NaN previos)
largo_unificado["potencia_mw"] = largo.set_index(["item_id","timestamp"])["potencia_mw"].reindex(
    pd.MultiIndex.from_frame(largo_unificado[["item_id","timestamp"]])
).values

# Capacidad nominal por planta
nominal = df_meteo_input.set_index("item_id")["pot_inst_mw"].to_dict()
largo_unificado["umbral_pot"] = largo_unificado["item_id"].map(nominal) * PCT_NOMINAL_MIN

# Agregar columna 'fecha' (día calendario)
largo_unificado["fecha"] = largo_unificado["timestamp"].dt.date

# Por cada (planta, día): contar horas con GHI alto, y cuántas de esas tienen pot~0
def estado_dia(grupo):
    sol_alto = grupo[grupo["ghi"] >= UMBRAL_GHI_ALTO]
    if len(sol_alto) == 0:
        return False   # día sin sol significativo, no se marca
    n_apagadas = (sol_alto["potencia_mw"] < sol_alto["umbral_pot"]).sum()
    return (n_apagadas / len(sol_alto)) >= PCT_HORAS_MIN

# Identificar días parados por planta
dias_parados = (largo_unificado
    .groupby(["item_id", "fecha"])
    .apply(estado_dia, include_groups=False)
)
dias_parados = dias_parados[dias_parados].reset_index().rename(columns={0: "parado"})
print(f"Total (planta, día) marcados como parada completa: {len(dias_parados):,}")
print(f"\nDesglose por planta:")
print(dias_parados.groupby("item_id").size().to_string())

# Marcar como NaN TODAS las filas (todas las horas) de los días parados
# Lookup vectorizado: ~100x más rápido que apply(axis=1).
parados_key_set = set(
    dias_parados["item_id"] + "__" + dias_parados["fecha"].astype(str)
)
mask_dia_parado = (
    largo_unificado["item_id"] + "__" + largo_unificado["fecha"].astype(str)
).isin(parados_key_set)
largo_unificado.loc[mask_dia_parado, "potencia_mw"] = np.nan

# Limpiar columnas auxiliares
largo_unificado = largo_unificado.drop(columns=["umbral_pot", "fecha"])

print(f"\n=== `largo_unificado` post-limpieza por días completos ===")
print(f"Shape: {largo_unificado.shape}")
print(f"Filas con NaN: {largo_unificado.potencia_mw.isna().sum():,}")
print(f"Filas válidas: {largo_unificado.potencia_mw.notna().sum():,}")
print(f"Días-planta marcados: {len(dias_parados):,}")
print(f"Horas perdidas: {largo_unificado.potencia_mw.isna().sum():,} (={largo_unificado.potencia_mw.isna().sum()/24:.0f} días-planta)")

In [ ]:
# === Inspección visual: 1 caja por planta, jerarquía visual clara ===
# Potencia y GHI: líneas sólidas en color (protagonistas).
# T, HR, Viento: líneas punteadas/rayadas en grises (cross-check contextual).
# Todas normalizadas a [0,1] dentro de la ventana para comparación.

import matplotlib.pyplot as plt
import pandas as pd

if "dias_parados" not in dir() or len(dias_parados) == 0:
    print("⚠ No hay 'dias_parados' en memoria. Re-corre sección 7.")
else:
    plantas_con_paradas = sorted(dias_parados["item_id"].unique())

    for planta in plantas_con_paradas:
        dias_planta = sorted(dias_parados[dias_parados.item_id == planta]["fecha"].tolist())
        dia_objetivo = dias_planta[0]
        print(f"\n{planta}: {len(dias_planta)} días parados. Plot del primero: {dia_objetivo}")

        ini = pd.Timestamp(dia_objetivo) - pd.Timedelta(days=3)
        fin = pd.Timestamp(dia_objetivo) + pd.Timedelta(days=4)

        muestra = largo_unificado[
            (largo_unificado.item_id == planta) &
            (largo_unificado.timestamp >= ini) &
            (largo_unificado.timestamp < fin)
        ].copy()

        pot_orig = largo[
            (largo.item_id == planta) &
            (largo.timestamp >= ini) &
            (largo.timestamp < fin)
        ].set_index("timestamp")["potencia_mw"]
        muestra = muestra.set_index("timestamp")
        muestra["potencia_mw"] = pot_orig
        muestra = muestra.reset_index().sort_values("timestamp")

        fechas_parados = set(dias_planta)

        # Variables con jerarquía visual:
        # protagonistas: color, sólido, grosor 2
        # contexto: gris/oscuro, estilos diferenciados, grosor 1, alpha bajo
        var_cfg = [
            # (col,           label,        color,         linestyle,  width, alpha)
            ("potencia_mw",   "Potencia",   "tab:blue",    "-",        2.2,   1.0),
            ("ghi",           "GHI",        "tab:orange",  "-",        2.0,   0.9),
            ("t_air",         "T aire",     "#666666",     "--",       1.1,   0.7),
            ("hr",            "HR",         "#666666",     ":",        1.3,   0.7),
            ("wind_speed",    "Viento",     "#666666",     "-.",       1.1,   0.7),
        ]

        fig, ax = plt.subplots(figsize=(14, 5.5))

        for col, label, color, ls, lw, al in var_cfg:
            serie = muestra[col]
            vmin, vmax = serie.min(), serie.max()
            if vmax > vmin:
                norm = (serie - vmin) / (vmax - vmin)
            else:
                norm = serie * 0
            ax.plot(
                muestra["timestamp"], norm,
                color=color, linestyle=ls, linewidth=lw, alpha=al,
                label=f"{label}  [{vmin:.1f}, {vmax:.1f}]",
            )

        # Sombrear días parados
        for fecha in muestra["timestamp"].dt.date.unique():
            if fecha in fechas_parados:
                ax.axvspan(
                    pd.Timestamp(fecha),
                    pd.Timestamp(fecha) + pd.Timedelta(days=1),
                    alpha=0.15, color="red",
                )

        ax.set_ylim(-0.05, 1.05)
        ax.set_ylabel("Valor normalizado a [0, 1]")
        ax.set_title(
            f"{planta} — ventana ±3 días alrededor de {dia_objetivo}  "
            f"(rojo: días NaN)\n"
            f"Sólidas (color): protagonistas para detectar parada operativa.  "
            f"Discontinuas (gris): cross-check meteorológico."
        )
        ax.legend(loc="upper right", ncol=5, fontsize=8, framealpha=0.92)
        ax.grid(alpha=0.25)
        plt.tight_layout()
        plt.show()

In [ ]:
# === Días con datos válidos por planta (post-limpieza de paradas) ===

resumen = []
for planta, g in largo_unificado.groupby("item_id"):
    g = g.sort_values("timestamp").copy()
    g["fecha"] = g["timestamp"].dt.date

    total_filas = len(g)
    filas_nan   = g["potencia_mw"].isna().sum()
    filas_ok    = total_filas - filas_nan

    dias_totales                = g["fecha"].nunique()
    dias_con_alguna_hora_valida = g[g["potencia_mw"].notna()]["fecha"].nunique()
    dias_completamente_nan      = dias_totales - dias_con_alguna_hora_valida

    nan_por_dia  = g.groupby("fecha")["potencia_mw"].apply(lambda s: s.isna().any())
    dias_limpios = (~nan_por_dia).sum()

    resumen.append({
        "item_id":             planta,
        "dias_totales":        dias_totales,
        "dias_limpios":        dias_limpios,
        "dias_con_alguna_NaN": dias_totales - dias_limpios,
        "dias_todo_NaN":       dias_completamente_nan,
        "horas_validas":       filas_ok,
        "horas_NaN":           filas_nan,
        "pct_dias_limpios":    round(100*dias_limpios/dias_totales, 1),
    })

df_dias = pd.DataFrame(resumen).sort_values("dias_totales", ascending=False).reset_index(drop=True)

from IPython.display import display
display(df_dias)

print(f"\n=== Totales ===")
print(f"Suma días-planta totales:           {df_dias.dias_totales.sum():,}")
print(f"Suma días-planta 100% limpios:      {df_dias.dias_limpios.sum():,}")
print(f"Suma días-planta con alguna NaN:    {df_dias.dias_con_alguna_NaN.sum():,}")
print(f"Suma días-planta con todo NaN:      {df_dias.dias_todo_NaN.sum():,}")

In [ ]:
# === Sección 8: Verificación de integridad temporal antes de P6 ===
# Confirma que `largo_unificado` tiene rejilla horaria sin huecos por planta.
# Detecta dos problemas que romperían AutoGluon TimeSeriesPredictor:
#   A) Timestamps faltantes en la rejilla (gaps temporales)
#   B) Timestamps duplicados (mismo (item_id, timestamp) dos veces)
#
# Si todo sale OK, P5 queda cerrado y `largo_unificado` está listo para P6.

import pandas as pd

print("=== Verificación de integridad temporal ===\n")

problemas_encontrados = []

for planta, g in largo_unificado.groupby("item_id"):
    g = g.sort_values("timestamp").reset_index(drop=True)

    # === A) Detectar gaps temporales ===
    diffs = g["timestamp"].diff().dropna()
    gaps = diffs[diffs != pd.Timedelta("1h")]
    n_gaps = len(gaps)
    gap_max = gaps.max() if n_gaps > 0 else pd.Timedelta(0)

    # === B) Detectar duplicados ===
    n_duplicados = g["timestamp"].duplicated().sum()

    # === C) Rejilla esperada vs real ===
    esperado = int((g.timestamp.max() - g.timestamp.min()).total_seconds() / 3600) + 1
    real = len(g)
    deficit = esperado - real

    estado = "✓ OK" if (n_gaps == 0 and n_duplicados == 0 and deficit == 0) else "⚠ PROBLEMA"

    print(f"{estado}  {planta:14}  "
          f"filas={real:>7,}  esperado={esperado:>7,}  "
          f"déficit={deficit:>5}  gaps={n_gaps:>4}  duplicados={n_duplicados:>3}")

    if n_gaps > 0 or n_duplicados > 0 or deficit > 0:
        problemas_encontrados.append({
            "planta": planta,
            "deficit_filas": deficit,
            "n_gaps": n_gaps,
            "gap_max": str(gap_max),
            "n_duplicados": n_duplicados,
        })

print(f"\n=== Resumen ===")
if not problemas_encontrados:
    print("✓ TODAS las plantas tienen rejilla horaria contigua y sin duplicados.")
    print("✓ `largo_unificado` está listo para AutoGluon en P6.")
else:
    print(f"⚠ {len(problemas_encontrados)} plantas con problemas:")
    print(pd.DataFrame(problemas_encontrados).to_string(index=False))
    print("\n→ Si hay déficit > 0: faltan timestamps en la rejilla. Hay que rellenarlos con NaN.")
    print("→ Si hay duplicados > 0: misma (item_id, timestamp) dos veces. Hay que de-duplicar.")
    print("→ Avísame y te paso celda de corrección.")

In [ ]:
# === Comparación de umbrales GHI sobre correlación de Spearman ===
import pandas as pd
import numpy as np

VARS = ["potencia_mw", "ghi", "t_air", "hr", "wind_speed", "pressure", "precip_mm"]
COVARS = VARS[1:]
UMBRALES = [0, 5, 10, 50, 100]

resultados = {}
n_filas = {}

for u in UMBRALES:
    df_u = largo_unificado[largo_unificado["ghi"] > u].copy()
    n_filas[u] = len(df_u)

    matrices_u = []
    for planta, g in df_u.groupby("item_id"):
        valid = g[VARS].dropna()
        if len(valid) < 100: continue
        matrices_u.append(valid.corr(method="spearman").loc["potencia_mw", COVARS].values)

    matrices_u = np.array(matrices_u)
    medianas = np.median(matrices_u, axis=0)
    resultados[u] = pd.Series(medianas, index=COVARS)

df_comp = pd.DataFrame(resultados).round(3)
df_comp.columns = [f"GHI>{u}" for u in UMBRALES]

print("=== Mediana de ρ(covariable, potencia_mw) por umbral GHI ===\n")
print(df_comp.to_string())

print("\n=== Tamaño de muestra por umbral ===")
for u in UMBRALES:
    pct = 100 * n_filas[u] / len(largo_unificado)
    print(f"  GHI > {u:>3}:  {n_filas[u]:>7,} filas  ({pct:5.1f}% del total)")

In [ ]:
#=== EDA final: heatmap mediana + violín/box/strip con GHI > 0 ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

VARS = ["potencia_mw", "ghi", "t_air", "hr", "wind_speed", "pressure", "precip_mm"]
COVARS = VARS[1:]

df = largo_unificado[largo_unificado["ghi"] > 0].copy()
print(f"Filas con GHI > 0: {len(df):,} de {len(largo_unificado):,} ({100*len(df)/len(largo_unificado):.1f}%)")

matrices = {}
for planta, g in df.groupby("item_id"):
    valid = g[VARS].dropna()
    matrices[planta] = valid.corr(method="spearman")

stack = np.stack([matrices[p].values for p in matrices], axis=0)
median_matrix = pd.DataFrame(np.median(stack, axis=0), index=VARS, columns=VARS)

# === Heatmap (sin cambios) ===
fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    median_matrix,
    annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, linecolor="gray",
    cbar_kws={"label": "Spearman ρ (mediana de 11 plantas)"},
    ax=ax,
)
ax.set_title(
    "Correlación de Spearman — mediana de las 11 plantas\n"
    "(filtrado a GHI > 0 W/m², NaN excluidos)"
)
plt.tight_layout()
plt.show()

# === Violin + Box + Strip plot horizontal con ajustes ===
filas_box = []
for planta, mat in matrices.items():
    for cov in COVARS:
        filas_box.append({
            "planta":     planta,
            "covariable": cov,
            "rho":        mat.loc["potencia_mw", cov],
        })
df_box = pd.DataFrame(filas_box)

orden = (df_box.assign(abs_rho=df_box["rho"].abs())
               .groupby("covariable")["abs_rho"].median()
               .sort_values(ascending=False).index.tolist())

fig, ax = plt.subplots(figsize=(11, 7.5))   # ← más alto (era 6.5)

sns.violinplot(
    data=df_box, y="covariable", x="rho",
    order=orden, inner=None,
    color="steelblue", alpha=0.25,
    cut=0, linewidth=0,
    ax=ax,
)
sns.boxplot(
    data=df_box, y="covariable", x="rho",
    order=orden, width=0.25,
    boxprops={"facecolor": "white", "edgecolor": "black", "linewidth": 1.2},
    medianprops={"color": "darkred", "linewidth": 2},
    whiskerprops={"color": "black", "linewidth": 1},
    capprops={"color": "black", "linewidth": 1},
    fliersize=0,
    ax=ax,
)
sns.stripplot(
    data=df_box, y="covariable", x="rho",
    order=orden, color="black", size=5, alpha=0.75,
    jitter=0.08,
    ax=ax,
)

ax.axvline(0, color="red", linestyle="--", alpha=0.5, linewidth=1)
ax.set_xlim(-0.85, 1.0)   # ← recortado para aprovechar espacio (antes era -1, 1)
ax.set_xlabel("Spearman ρ vs. potencia_mw")
ax.set_ylabel("")
ax.set_title(
    "Distribución de ρ(covariable, potencia) en las 11 plantas\n"
    "violín = densidad KDE | caja = IQR + mediana | punto = una planta\n"
    "(filtro: GHI > 0 W/m², orden por |ρ| mediana desc.)"
)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

resumen = df_box.groupby("covariable")["rho"].agg(["median", "mean", "std", "min", "max"]).round(3)
resumen = resumen.loc[orden]
print("\n=== Resumen ρ(covariable, potencia) entre las 11 plantas (GHI > 0) ===")
print(resumen.to_string())

In [ ]:
# === Sección 9a: Descarga automática de fichas técnicas COES (flujo 2 pasos) ===
import os, time
import requests
from pathlib import Path

BASE = "https://www.coes.org.pe/Portal/equipamiento/fichatecnica"
URL_EXPORT = f"{BASE}/ExportarFichaTecnica"
URL_DESCARGAR = f"{BASE}/DescargarExcel"
URL_DETALLE = f"{BASE}/IndexDetalle"

OUT_DIR = Path("data/raw/fichas")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PLANTAS = {
    "CARHUAQUERO":  23437,
    "CLEMESI":      22189,
    "INTIPAMPA":    17796,
    "MAJES":        13402,
    "MATARANI":     23256,
    "MOQUEGUA_FV":  14762,
    "PANAMERICANA": 13533,
    "REPARTICION":  13399,
    "RUBI":         18074,
    "TACNA":        13503,
    "YARUCAYA":     21771,
}

def es_excel(r):
    if r.status_code != 200 or len(r.content) < 2000:
        return False
    return r.content[:2] == b"PK" or r.content[:2] == b"\xd0\xcf"

def bajar_ficha(s, id_elemento):
    """Flujo de 2 pasos: prepara archivo en server (POST) y luego lo descarga (GET)."""
    form = {"Content-Type": "application/x-www-form-urlencoded; charset=UTF-8"}
    s.headers["Referer"] = f"{URL_DETALLE}?idFicha=35&idElemento={id_elemento}"

    # Paso 1: prepara el archivo en TEMP del server
    data = {
        "idFicha": "35",
        "idElemento": str(id_elemento),
        "opcionComentario": "2",
    }
    r1 = s.post(URL_EXPORT, headers=form, data=data, timeout=120)
    if r1.status_code != 200:
        return None, f"paso1 HTTP {r1.status_code}"

    # El JSON devuelve: [ruta_temp_server, nombre_archivo_sugerido]
    try:
        archivo, nombre = r1.json()
    except Exception as e:
        return None, f"paso1 no es JSON: {r1.text[:100]}"

    # Paso 2: descarga real con esos dos parámetros
    r2 = s.get(URL_DESCARGAR, params={"archivo": archivo, "nombre": nombre}, timeout=120)
    return r2, None

# === Ejecución ===
s = make_session()
print("Sesión inicial creada.\n")

resultados = []
for planta, id_elem in PLANTAS.items():
    print(f"→ {planta:14} (idElem={id_elem})... ", end="", flush=True)
    try:
        r, err = bajar_ficha(s, id_elem)
        if err:
            print(f"✗ {err}")
            resultados.append({"planta": planta, "ok": False, "razon": err})
            continue
        if not es_excel(r):
            print(f"✗ no es Excel (HTTP {r.status_code}, {len(r.content)} bytes)")
            resultados.append({"planta": planta, "ok": False, "razon": "no es Excel"})
            continue
        ext = "xlsx" if r.content[:2] == b"PK" else "xls"
        out = OUT_DIR / f"{planta}.{ext}"
        out.write_bytes(r.content)
        print(f"✓ {ext} ({len(r.content)/1024:.1f} KB)")
        resultados.append({"planta": planta, "ok": True, "path": str(out)})
    except Exception as e:
        print(f"✗ {e}")
        resultados.append({"planta": planta, "ok": False, "razon": str(e)})
    time.sleep(1.5)

import pandas as pd
df_res = pd.DataFrame(resultados)
n_ok = df_res["ok"].sum()
print(f"\n=== {n_ok}/11 descargadas ===")
if n_ok < 11:
    print(df_res[~df_res.ok].to_string(index=False))

In [ ]:
# === Sección 9b: Construcción y limpieza de df_static desde fichas COES ===
# Parsea hojas "CENTRALES" y "COMPONENTE DE CENTRAL SOLAR", canoniza categorías,
# y agrega lat/lon desde df_meteo_input. Listo para AutoGluon static_features.

# Instalar openpyxl si no está (necesario para leer .xlsx)
%pip install openpyxl -q

import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display

FICHAS_DIR = Path("data/raw/fichas")

PLANTAS = [
    "RUBI", "PANAMERICANA", "TACNA", "MAJES", "MOQUEGUA_FV",
    "REPARTICION", "INTIPAMPA", "YARUCAYA", "CLEMESI",
    "CARHUAQUERO", "MATARANI",
]

# === Helpers de extracción ===

def _norm(x):
    """Normaliza strings, maneja 'No Aplica', 'Sin Datos', vacíos."""
    if pd.isna(x): return None
    s = str(x).strip()
    if s.lower() in ("no aplica", "sin datos", "n/a", "na", "", "nan", "none"):
        return None
    return s

def _to_float(x):
    """Convierte a float extrayendo el primer número del texto."""
    s = _norm(x)
    if s is None: return None
    s = s.replace(",", "")
    m = re.search(r"-?\d+\.?\d*", s)
    if m:
        try: return float(m.group())
        except: return None
    return None

def _to_int(x):
    f = _to_float(x)
    return int(f) if f is not None else None

def _extract_max_pct(x):
    """De 'Max: 99.3%;EURO: 98.69%' extrae 99.3."""
    s = _norm(x)
    if s is None: return None
    m = re.search(r"(\d+\.?\d*)\s*%?", s)
    return float(m.group(1)) if m else None

def buscar_fila(df, keywords_etiqueta):
    """Busca primera fila con TODAS las keywords; devuelve última celda no-nula."""
    for _, row in df.iterrows():
        valores = [c for c in row.values if pd.notna(c)]
        texto = " ".join(str(v) for v in valores).lower()
        if all(kw.lower() in texto for kw in keywords_etiqueta):
            if valores: return valores[-1]
    return None

def parsear_ficha(path):
    """Lee un Excel COES y devuelve dict con campos relevantes (sin coef_temp)."""
    xl = pd.ExcelFile(path)
    hojas = xl.sheet_names

    hoja_c_nom = next((h for h in hojas if "CENTRAL" in h.upper() and "COMPONENTE" not in h.upper()), None)
    hoja_k_nom = next((h for h in hojas if "COMPONENTE" in h.upper()), None)

    if hoja_c_nom is None:
        raise ValueError(f"No se encontró hoja CENTRALES. Hojas: {hojas}")

    df_c = pd.read_excel(xl, sheet_name=hoja_c_nom, header=None)
    df_k = pd.read_excel(xl, sheet_name=hoja_k_nom, header=None) if hoja_k_nom else None

    out = {
        "pot_inst_mw":      _to_float(buscar_fila(df_c, ["Potencia nominal de la central"])),
        "tracking":         _norm(buscar_fila(df_c, ["Tecnología del seguimiento"])),
        "n_modulos":        _to_int(buscar_fila(df_c, ["Número total de módulos"])),
        "nivel_mt_kv":      _to_float(buscar_fila(df_c, ["Nivel de media tensión"])),
        "horas_eq_anuales": _to_int(buscar_fila(df_c, ["Horas de utilización", "anual"])),
    }

    if df_k is not None:
        # Aislar secciones MÓDULO vs INVERSOR
        modulo_ini = inversor_ini = None
        for i, row in df_k.iterrows():
            texto = " ".join(str(c) for c in row.values if pd.notna(c)).upper()
            if modulo_ini is None and "MÓDULO" in texto:
                modulo_ini = i
            elif modulo_ini is not None and inversor_ini is None and "INVERSOR" in texto:
                inversor_ini = i
                break
        df_mod = df_k.iloc[modulo_ini:inversor_ini] if modulo_ini is not None else df_k
        df_inv = df_k.iloc[inversor_ini:] if inversor_ini is not None else df_k

        out["modulo_fabricante"]   = _norm(buscar_fila(df_mod, ["Fabricante"]))
        out["modulo_tecnologia"]   = _norm(buscar_fila(df_mod, ["Tecnología"]))
        out["inversor_fabricante"] = _norm(buscar_fila(df_inv, ["Fabricante"]))
        out["inversor_rendim_max"] = _extract_max_pct(buscar_fila(df_inv, ["Rendimiento"]))
    else:
        for k in ["modulo_fabricante", "modulo_tecnologia", "inversor_fabricante", "inversor_rendim_max"]:
            out[k] = None

    return out

# === Canonizadores ===

def canon_tracking(s):
    """Canoniza variantes léxicas a 3 categorías: fijo, eje_horizontal, eje_vertical."""
    if pd.isna(s): return None
    sl = str(s).lower()
    if "fij" in sl:
        return "fijo"
    if "vertical" in sl:
        return "eje_vertical"
    if "horizontal" in sl or "horiz" in sl or "1 eje" in sl or "un eje" in sl:
        return "eje_horizontal"
    return None

def canon_tecnologia(s):
    """Canoniza tecnología de módulo a categorías limpias."""
    if pd.isna(s): return None
    sl = str(s).lower()
    if "mono" in sl and "amorfo" not in sl:
        return "mono_si"
    if "amorfo" in sl:
        return "mixto_aSi_polySi"
    if "poli" in sl or "policri" in sl:
        return "poly_si"
    if "bifacial" in sl: return "bifacial"
    if "monofacial" in sl: return "monofacial"
    if "tipo p" in sl or "tipo_p" in sl: return "tipo_p"
    return None

# === Construir df_static ===

filas = []
for planta in PLANTAS:
    candidatos = list(FICHAS_DIR.glob(f"{planta}.xls*"))
    if not candidatos:
        print(f"⚠ Falta archivo de {planta}")
        continue
    try:
        datos = parsear_ficha(candidatos[0])
        datos["item_id"] = planta
        filas.append(datos)
        print(f"✓ {planta}")
    except Exception as e:
        print(f"✗ {planta}: {e}")

df_static = pd.DataFrame(filas)

# Aplicar canonización
df_static["tracking"]          = df_static["tracking"].apply(canon_tracking)
df_static["modulo_tecnologia"] = df_static["modulo_tecnologia"].apply(canon_tecnologia)

# Reordenar columnas (sin modulo_coef_temp por decisión cerrada)
cols_order = [
    "item_id", "pot_inst_mw", "tracking", "n_modulos",
    "modulo_tecnologia", "inversor_rendim_max",
]
df_static = df_static[cols_order]

# Agregar latitud y longitud desde df_meteo_input
df_static = df_static.merge(
    df_meteo_input[["item_id", "lat", "lon"]].rename(columns={"lat": "latitude", "lon": "longitude"}),
    on="item_id", how="left",
)

# === Diagnóstico final ===
print(f"\n=== df_static construido ===")
print(f"Shape: {df_static.shape}")

print(f"\nDiagnóstico de nulls por columna:")
for col in df_static.columns:
    n = df_static[col].isna().sum()
    flag = " ⚠" if n > 0 else ""
    print(f"  {col:25} nulls={n}/11{flag}")

print(f"\nCategorías canonizadas — tracking: {df_static.tracking.value_counts(dropna=False).to_dict()}")
print(f"Categorías canonizadas — modulo_tecnologia: {df_static.modulo_tecnologia.value_counts(dropna=False).to_dict()}")

display(df_static)

In [ ]:
# === Agregar altitud vía API Open-Elevation ===
import requests
import time

def obtener_elevacion(lat, lon, retry=2):
    """Consulta API Open-Elevation. Devuelve elevación en metros o None si falla."""
    url = "https://api.open-elevation.com/api/v1/lookup"
    for intento in range(retry + 1):
        try:
            r = requests.get(url, params={"locations": f"{lat},{lon}"}, timeout=20)
            if r.status_code == 200:
                return r.json()["results"][0]["elevation"]
        except Exception as e:
            if intento < retry:
                time.sleep(2)
                continue
            print(f"  ⚠ fallo en lat={lat}, lon={lon}: {e}")
    return None

print(f"\n=== Obteniendo altitud para las 11 plantas (Open-Elevation API) ===")
elevaciones = []
for _, row in df_static.iterrows():
    elev = obtener_elevacion(row["latitude"], row["longitude"])
    elevaciones.append(elev)
    print(f"  {row['item_id']:14}  lat={row['latitude']:8.4f}  lon={row['longitude']:8.4f}  alt={elev} m")
    time.sleep(0.5)   # rate limit cortés

df_static["altitude_m"] = elevaciones

In [ ]:
import torch
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:   {torch.cuda.get_device_name(0)}")
    print(f"VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠ CUDA no detectada — revisar instalación de torch.")

In [ ]:
import pandas as pd
import autogluon.timeseries

from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

In [ ]:
train_data = TimeSeriesDataFrame.from_data_frame(
    largo_unificado,
    id_column="item_id",
    timestamp_column="timestamp",
    static_features_df=df_static,
)
train_data.head(48)

In [ ]:
# 2. NaN en las últimas 168h de cada planta
print("\n=== NaN en últimas 168h por planta ===")
problemas = []
for planta in train_data.item_ids:
    serie = train_data.loc[planta]
    ultimas_168 = serie.tail(168)
    n_nan = ultimas_168["potencia_mw"].isna().sum()
    flag = " ⚠" if n_nan > 0 else " ✓"
    print(f"  {planta:14} NaN: {n_nan:>3}{flag}")
    if n_nan > 0:
        problemas.append((planta, n_nan))

if problemas:
    print(f"\n⚠ {len(problemas)} plantas con NaN en holdout (manejado automáticamente por AutoGluon).")
else:
    print(f"\n✓ Todas las plantas con target completo en holdout.")


In [ ]:
print("=== NaN en covariables (NO debería haber) ===")
covars = ["ghi", "t_air", "hr", "wind_speed", "pressure", "precip_mm"]
for c in covars:
    n = train_data[c].isna().sum()
    flag = " ⚠ PROBLEMA" if n > 0 else " ✓"
    print(f"  {c:15} NaN={n:>5}{flag}")

print(f"\n=== NaN en target (OK que existan) ===")
print(f"  potencia_mw    NaN={train_data['potencia_mw'].isna().sum():>5}  (paradas detectadas)")

In [ ]:
# === Custom metric: Pearson R (para diagrama de Taylor) ===
import numpy as np
import pandas as pd
from autogluon.timeseries.metrics import TimeSeriesScorer

class PearsonR(TimeSeriesScorer):
    """
    Pearson correlation coefficient sobre la predicción puntual (mediana).
    Calculado por serie y promediado. Solo para reporte, NO como eval_metric principal.
    """
    needs_quantile = False
    greater_is_better_internal = True
    optimum = 1.0

    def compute_metric(self, data_future, predictions, target, **kwargs):
        y_true = data_future[target]
        y_pred = predictions["mean"]

        rs = []
        for item_id in y_true.index.get_level_values("item_id").unique():
            y_t = y_true.loc[item_id].values
            y_p = y_pred.loc[item_id].values

            mask = ~(np.isnan(y_t) | np.isnan(y_p))
            if mask.sum() < 2:
                continue
            y_t = y_t[mask]
            y_p = y_p[mask]

            if y_t.std() == 0 or y_p.std() == 0:
                continue

            r = np.corrcoef(y_t, y_p)[0, 1]
            rs.append(r)

        return float(np.mean(rs)) if rs else 0.0

pearson_r_metric = PearsonR()
print(f"✓ Custom metric PearsonR definida")

In [ ]:
# === SmartPersistenceModel: baseline solar con clear-sky correction ===
import logging
from typing import Optional, Tuple

import numpy as np
import pandas as pd
import pvlib
from scipy.stats import norm

from autogluon.timeseries import TimeSeriesDataFrame
from autogluon.timeseries.models.abstract import AbstractTimeSeriesModel


class SmartPersistenceModelV2(AbstractTimeSeriesModel):
    """
    Smart Persistence baseline para pronóstico solar fotovoltaico day-ahead.

    Fórmula: P̂(t+h) = P(t_ref) × CSGHI(t+h) / CSGHI(t_ref)

    donde CSGHI se calcula con el modelo clear-sky Ineichen-Perez vía pvlib,
    usando coordenadas (lat, lon) y altitud por planta desde static_features.

    Referencias:
        - Ineichen P., Perez R. (2002). "A new airmass independent formulation
          for the Linke turbidity coefficient." Solar Energy 73(3): 151–157.
        - Yang D. (2019). "Making reference solar forecasts with climatology,
          persistence, and their optimal convex combination." Solar Energy 193: 981-985.
    """

    _supports_known_covariates: bool = False
    _supports_past_covariates: bool = False
    _supports_static_features: bool = True

    # Banda de incertidumbre heurística: sigma proporcional a P_ref y al horizonte
    _SIGMA_BASE = 0.10   # 10% de P_ref a h=1
    _SIGMA_GROW = 0.005  # +0.5% adicional por hora de horizonte
    _TZ = "America/Lima"

    def _fit(
        self,
        train_data: TimeSeriesDataFrame,
        val_data: Optional[TimeSeriesDataFrame] = None,
        time_limit: Optional[float] = None,
        **kwargs,
    ) -> None:
        """
        Smart Persistence NO requiere entrenamiento estadístico.
        Solo cachea pvlib.Location por planta usando static_features.
        """
        self.locations = {}
        static = train_data.static_features

        if static is None:
            raise ValueError(
                "SmartPersistenceModel requiere static_features con columnas "
                "[latitude, longitude, altitude_m]"
            )

        required_cols = {"latitude", "longitude", "altitude_m"}
        missing = required_cols - set(static.columns)
        if missing:
            raise ValueError(
                f"static_features carece de columnas requeridas: {missing}"
            )

        for item_id, row in static.iterrows():
            altitude = row["altitude_m"] if pd.notna(row["altitude_m"]) else 0.0
            self.locations[item_id] = pvlib.location.Location(
                latitude=float(row["latitude"]),
                longitude=float(row["longitude"]),
                tz=self._TZ,
                altitude=float(altitude),
                name=str(item_id),
            )

    def _predict(
        self,
        data: TimeSeriesDataFrame,
        known_covariates: Optional[TimeSeriesDataFrame] = None,
        **kwargs,
    ) -> TimeSeriesDataFrame:

        quantile_levels = self.quantile_levels
        H = self.prediction_length

        predictions_list = []

        for item_id in data.item_ids:
            serie = data.loc[item_id]
            target_col = self.target
            pot_series = serie[target_col].dropna()
            loc = self.locations.get(item_id)

            t_last = serie.index.get_level_values("timestamp")[-1]
            future_times_naive = pd.date_range(
                start=t_last + pd.Timedelta(hours=1),
                periods=H,
                freq=self.freq,
            )

            point_pred = np.zeros(H)

            for h_idx, t_fut in enumerate(future_times_naive):
                t_ref = t_fut - pd.Timedelta(hours=24)

                if t_ref in pot_series.index:
                    P_ref = float(pot_series.loc[t_ref])
                else:
                    candidates = pot_series[pot_series.index <= t_ref]
                    P_ref = float(candidates.iloc[-1]) if len(candidates) > 0 else 0.0

                if loc is None or P_ref <= 0:
                    point_pred[h_idx] = max(P_ref, 0.0)
                    continue

                try:
                    t_ref_tz = pd.Timestamp(t_ref).tz_localize(self._TZ)
                    t_fut_tz = pd.Timestamp(t_fut).tz_localize(self._TZ)
                    cs_ref = loc.get_clearsky(pd.DatetimeIndex([t_ref_tz]), model="ineichen")["ghi"].iloc[0]
                    cs_fut = loc.get_clearsky(pd.DatetimeIndex([t_fut_tz]), model="ineichen")["ghi"].iloc[0]
                except Exception:
                    point_pred[h_idx] = max(P_ref, 0.0)
                    continue

                if cs_ref < 1.0:
                    point_pred[h_idx] = max(P_ref, 0.0)
                else:
                    point_pred[h_idx] = max(P_ref * cs_fut / cs_ref, 0.0)

            future_idx = pd.MultiIndex.from_product(
                [[item_id], future_times_naive],
                names=["item_id", "timestamp"],
            )

            item_pred = pd.DataFrame(
                {
                    "mean": point_pred,
                    "0.5": point_pred,
                    **{str(q): point_pred for q in quantile_levels},
                },
                index=future_idx,
            )
            predictions_list.append(item_pred)

        return TimeSeriesDataFrame(pd.concat(predictions_list))

In [ ]:
from autogluon.timeseries.configs.hyperparameter_presets import get_hyperparameter_presets
wf_hp_check = get_hyperparameter_presets()["default"]
print(type(wf_hp_check.get("DirectTabular")))
print(wf_hp_check.get("DirectTabular"))

In [ ]:
import os
os.environ["AUTOGLUON_MEMORY_LIMIT_MB"] = "14000"
os.environ["HF_HUB_OFFLINE"] = "1"

# === WALK-FORWARD best_quality: refit mensual expanding + predicción diaria, TODOS los modelos ===
import pandas as pd, numpy as np, logging, shutil
from pathlib import Path
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
from autogluon.timeseries.configs.hyperparameter_presets import get_hyperparameter_presets

PREDICTION_LENGTH = 24
HIST_MIN_DIAS     = 365
STRIDE_DIAS       = 90
TIME_LIMIT_FIT    = None
COVARS = ["ghi","t_air","hr","wind_speed","pressure","precip_mm"]

WF_HP    = get_hyperparameter_presets()["default"]
WF_DIR   = Path("models/walkforward_v1")
FIT_DIR  = WF_DIR / "fits";  FIT_DIR.mkdir(parents=True, exist_ok=True)
PRED_DIR = WF_DIR / "preds"; PRED_DIR.mkdir(parents=True, exist_ok=True)

base = largo_unificado.sort_values(["item_id","timestamp"]).reset_index(drop=True)
t_min, t_max = base["timestamp"].min().normalize(), base["timestamp"].max().normalize()
dias = pd.date_range(t_min + pd.Timedelta(days=HIST_MIN_DIAS),
                     t_max - pd.Timedelta(days=1), freq="D")
# fecha de entrada de cada planta = su primer día + mínimo de historia (mismo umbral para todas)
entrada = base.groupby("item_id")["timestamp"].min().dt.normalize() + pd.Timedelta(days=HIST_MIN_DIAS)
print(f"Ventanas day-ahead: {len(dias)}  ({dias[0].date()} → {dias[-1].date()})")

def tsdf(df_):
    return TimeSeriesDataFrame.from_data_frame(
        df_, id_column="item_id", timestamp_column="timestamp", static_features_df=df_static)

def _liberar_y_borrar(p):                     # cierra handlers de logging y borra (Windows)
    for lg in [logging.getLogger(), *logging.Logger.manager.loggerDict.values()]:
        for h in list(getattr(lg, "handlers", [])):
            if isinstance(h, logging.FileHandler) and p.name in str(getattr(h, "baseFilename", "")):
                h.close(); lg.removeHandler(h)
    shutil.rmtree(p, ignore_errors=True)

predictor_wf, ultimo_refit, modelos = None, None, None
for i, dia in enumerate(dias):
    ok = entrada[entrada <= dia].index                 # plantas con >= HIST_MIN_DIAS a la fecha
    if len(ok) == 0:
        continue

    es_refit = (predictor_wf is None) or ((dia - ultimo_refit).days >= STRIDE_DIAS)
    if es_refit:
        fit_path = FIT_DIR / f"fit_{dia:%Y%m%d}"
        predictor_file = fit_path / "predictor.pkl"
        ctx_df = base[(base["timestamp"] < dia) & base["item_id"].isin(ok)]
        if predictor_file.exists():
            predictor_wf = TimeSeriesPredictor.load(str(fit_path))
            print(f"[{i:4}/{len(dias)}] LOAD  {dia.date()} (refit cacheado)")
        else:
            if fit_path.exists():
                _liberar_y_borrar(fit_path)
            predictor_wf = TimeSeriesPredictor(
                prediction_length=PREDICTION_LENGTH, target="potencia_mw",
                eval_metric="SQL", eval_metric_seasonal_period=24,
                known_covariates_names=COVARS, freq="h",
                path=str(fit_path), verbosity=1)
            predictor_wf.fit(train_data=tsdf(ctx_df), hyperparameters=WF_HP,
                             num_val_windows="auto", refit_every_n_windows="auto",
                             time_limit=TIME_LIMIT_FIT, enable_ensemble=True)
            print(f"[{i:4}/{len(dias)}] REFIT {dia.date()} ({len(ok)} plantas, ctx={len(ctx_df):,})")
        ultimo_refit, modelos = dia, predictor_wf.model_names()

    out_path = PRED_DIR / f"{dia:%Y%m%d}.parquet"
    if out_path.exists():
        continue

    ctx_df = base[(base["timestamp"] < dia) & base["item_id"].isin(ok)]
    fut_df = base[(base["timestamp"] >= dia) & (base["timestamp"] < dia + pd.Timedelta(days=1))
                  & base["item_id"].isin(ok)]
    if fut_df.empty:
        continue

    ctx_ts   = tsdf(ctx_df)
    fut_covs = tsdf(fut_df[["item_id","timestamp"]+COVARS])
    act      = fut_df[["item_id","timestamp","potencia_mw"]]

    dia_preds = []
    for m in modelos:
        try:
            pm = predictor_wf.predict(ctx_ts, known_covariates=fut_covs, model=m).reset_index()
            pm["model"] = m
            dia_preds.append(pm)
        except Exception as e:
            print(f"   ⚠ {dia.date()} {m}: {e}")
    if dia_preds:
        P = pd.concat(dia_preds, ignore_index=True).merge(act, on=["item_id","timestamp"], how="left")
        P["dia"] = dia
        P.to_parquet(out_path)

print("\n✓ Walk-forward terminado (o reanudable). Predicciones por día en", PRED_DIR)

In [ ]:
import autogluon.core as ag
print(ag.__version__)

In [ ]:
# === Evaluación honesta del walk-forward ===
import pandas as pd
import numpy as np
from pathlib import Path

PRED_DIR = Path("models/walkforward_v1/preds")
CAPACIDADES_MW = df_static.set_index("item_id")["pot_inst_mw"].to_dict()
QUANTILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

archivos = sorted(PRED_DIR.glob("*.parquet"))
print(f"Días de predicción encontrados: {len(archivos)}")
df = pd.concat([pd.read_parquet(f) for f in archivos], ignore_index=True)
print(f"Filas totales: {len(df):,}")

resultados = []
for modelo, gm in df.groupby("model"):
    for planta, gp in gm.groupby("item_id"):
        cap = CAPACIDADES_MW.get(planta)
        if cap is None:
            continue

        y_true = gp["potencia_mw"].values
        y_med  = gp["0.5"].values
        y_mean = gp["mean"].values
        mask   = ~np.isnan(y_true) & ~np.isnan(y_med)
        if mask.sum() == 0:
            continue

        yt = y_true[mask]
        ym = y_med[mask]
        ymean = y_mean[mask]

        # Métricas punto (mediana)
        mae  = np.mean(np.abs(yt - ym))
        rmse = np.sqrt(np.mean((ymean - yt) ** 2))

        # WQL — Pinball loss promediado sobre quantiles
        pinballs = []
        for q in QUANTILES:
            yq = gp[str(q)].values[mask]
            pb = np.mean(np.where(yt >= yq, q * (yt - yq), (1 - q) * (yq - yt)))
            pinballs.append(pb)
        wql = 2 * np.mean(pinballs) / (np.mean(yt) + 1e-8)

        # SQL — Scaled quantile loss (escala por MAE naive estacional)
        # Aproximación: seasonal naive con periodo 24h
        y_naive = gp["potencia_mw"].shift(24).values[mask]
        mask2 = ~np.isnan(y_naive)
        scale = np.mean(np.abs(yt[mask2] - y_naive[mask2])) + 1e-8
        sql = 2 * np.mean(pinballs) / scale

        r = np.corrcoef(yt, ym)[0, 1]

        resultados.append({
            "model":   modelo,
            "item_id": planta,
            "nMAE":    mae / cap,
            "nRMSE":   rmse / cap,
            "WQL":     wql,
            "SQL":     sql,
            "R":       r,
            "n":       mask.sum()
        })

df_res = pd.DataFrame(resultados)

resumen_mean = df_res.groupby("model")[["nMAE","nRMSE","WQL","SQL","R"]].mean().sort_values("nMAE")
print("\n=== Métricas por modelo (walk-forward honesto, media entre plantas) ===")
print(resumen_mean.round(4))

resumen = df_res.groupby("model")[["nMAE","nRMSE","WQL","SQL","R"]].median().sort_values("nMAE")
print("\n=== Métricas por modelo (walk-forward honesto, mediana entre plantas) ===")
print(resumen.round(4))

In [ ]:
print(pd.read_parquet(archivos[0]).columns.tolist())
print(df_res.sort_values(["item_id","nMAE"]).to_string())

In [ ]:
import autogluon.timeseries
print(f"AutoGluon version: {autogluon.timeseries.__version__}")

# Ver TODOS los presets disponibles
import pathlib
ag_path = pathlib.Path(autogluon.timeseries.__file__).parent

for f in ag_path.rglob("*.py"):
    text = f.read_text(errors="ignore")
    if '"best_quality"' in text or "'best_quality'" in text:
        print(f"\n=== Encontrado en: {f} ===")
        # Imprimir contexto del match
        lines = text.split("\n")
        for i, line in enumerate(lines):
            if "best_quality" in line:
                start = max(0, i - 2)
                end = min(len(lines), i + 20)
                for j in range(start, end):
                    print(f"  {j:4} | {lines[j]}")
                print("  ---")

In [ ]:
from pathlib import Path
import pandas as pd

PRED_DIR = Path("models/walkforward_v1/preds")
archivos = sorted(PRED_DIR.glob("*.parquet"))
df_pred = pd.concat([pd.read_parquet(f) for f in archivos], ignore_index=True)
print(df_pred.shape, df_pred["model"].unique() if "model" in df_pred.columns else "sin columna model")

In [ ]:
import numpy as np, pandas as pd

# 1. Ensemble por fila: de df_pred, filtrado a WeightedEnsemble
ens = df_pred[df_pred["model"] == "WeightedEnsemble"][["item_id","timestamp","0.5","potencia_mw"]].copy() \
      if "model" in df_pred.columns else \
      df_pred[["item_id","timestamp","0.5","potencia_mw"]].copy()
ens = ens.rename(columns={"0.5": "pred_ens"})

# 2. SmartPersistence por fila: reconstruir con sp_pred por planta
#    (mismo cálculo del loop, reteniendo el detalle en vez de solo el resumen)
sp_rows = []
static_idx = df_static.set_index("item_id")
for planta, gp in largo_unificado.groupby("item_id"):
    if planta not in static_idx.index:
        continue
    lat = static_idx.loc[planta,"latitude"]; lon = static_idx.loc[planta,"longitude"]
    alt = static_idx.loc[planta,"altitude_m"]
    gp = gp.sort_values("timestamp").set_index("timestamp")
    import pvlib
    loc = pvlib.location.Location(lat, lon, altitude=alt)
    cs = loc.get_clearsky(gp.index, model="ineichen")
    gp["cs_ghi"] = cs["ghi"]
    gp["P_ref"]  = gp["potencia_mw"].shift(24)
    gp["cs_ref"] = gp["cs_ghi"].shift(24)
    gp["sp_pred"] = np.where(gp["cs_ref"] >= 1.0,
                             (gp["P_ref"]*gp["cs_ghi"]/gp["cs_ref"]).clip(lower=0),
                             gp["P_ref"].clip(lower=0))
    tmp = gp.reset_index()[["timestamp","sp_pred"]]
    tmp["item_id"] = planta
    sp_rows.append(tmp)
sp = pd.concat(sp_rows, ignore_index=True)

# 3. INNER JOIN: misma muestra para ambos
m = ens.merge(sp, on=["item_id","timestamp"], how="inner").dropna(
        subset=["pred_ens","sp_pred","potencia_mw"])

# 4. nMAE de AMBOS sobre la MISMA muestra, por planta, luego mediana
CAP = df_static.set_index("item_id")["pot_inst_mw"].to_dict()
rows = []
for planta, g in m.groupby("item_id"):
    cap = CAP.get(planta)
    if cap is None or len(g) < 100:
        continue
    yt = g["potencia_mw"].values
    rows.append({
        "item_id": planta,
        "nmae_ens": np.mean(np.abs(yt - g["pred_ens"].values))/cap,
        "nmae_sp":  np.mean(np.abs(yt - g["sp_pred"].values))/cap,
        "nrmse_ens": np.sqrt(np.mean((yt - g["pred_ens"].values)**2))/cap,
        "nrmse_sp":  np.sqrt(np.mean((yt - g["sp_pred"].values)**2))/cap,
        "n": len(g),
    })
ss = pd.DataFrame(rows)

SS_nMAE  = 1 - ss["nmae_ens"].median()/ss["nmae_sp"].median()
SS_nRMSE = 1 - ss["nrmse_ens"].median()/ss["nrmse_sp"].median()
print(f"nMAE  Ensemble={ss['nmae_ens'].median():.4f}  SP={ss['nmae_sp'].median():.4f}")
print(f"nRMSE Ensemble={ss['nrmse_ens'].median():.4f}  SP={ss['nrmse_sp'].median():.4f}")
print(f"SS_nMAE  auditado = {SS_nMAE:.4f}")
print(f"SS_nRMSE auditado = {SS_nRMSE:.4f}")
print(f"Muestra común: {ss['n'].sum():,} pares, {len(ss)} plantas")

In [ ]:
print("df_pred cols:", df_pred.columns.tolist())
print("¿tiene 'model'?", "model" in df_pred.columns)
if "model" in df_pred.columns:
    print("modelos:", df_pred["model"].unique())

In [ ]:
# === Diagnóstico autocontenido: por qué SP pierde filas vs. el walk-forward ===
# Recalcula todo desde cero para una sola planta, sin depender de variables
# que el loop de la celda 46 haya dejado en un estado de la última iteración.

import pvlib
import numpy as np
import pandas as pd

def diagnosticar_planta(planta, largo_unificado, df_static, claves_wf):
    static_idx = df_static.set_index("item_id")

    lat = static_idx.loc[planta, "latitude"]
    lon = static_idx.loc[planta, "longitude"]
    alt = static_idx.loc[planta, "altitude_m"]

    gp = (
        largo_unificado[largo_unificado.item_id == planta]
        .sort_values("timestamp")
        .set_index("timestamp")
        .copy()
    )

    loc = pvlib.location.Location(lat, lon, altitude=alt)
    cs = loc.get_clearsky(gp.index, model="ineichen")
    gp["cs_ghi"] = cs["ghi"]

    gp["P_ref"] = gp["potencia_mw"].shift(24)
    gp["cs_ref"] = gp["cs_ghi"].shift(24)

    gp["sp_pred"] = np.where(
        gp["cs_ref"] >= 1.0,
        (gp["P_ref"] * gp["cs_ghi"] / gp["cs_ref"]).clip(lower=0),
        gp["P_ref"].clip(lower=0),
    )

    # Reset index para poder hacer merge limpio por columna "timestamp"
    gp_reset = gp.reset_index()

    claves_planta = claves_wf[claves_wf["item_id"] == planta][["timestamp"]]
    gp_match = gp_reset.merge(claves_planta, on="timestamp", how="inner")

    mask_eval = (
        ~gp_match["potencia_mw"].isna()
        & ~gp_match["sp_pred"].isna()
        & ~gp_match["P_ref"].isna()
    )

    perdidas = gp_match[~mask_eval]

    print(f"=== {planta} ===")
    print(f"Filas en común con walk-forward: {len(gp_match):,}")
    print(f"Filas válidas (mask_eval=True):   {mask_eval.sum():,}")
    print(f"Filas perdidas (mask_eval=False): {len(perdidas):,}")
    print()
    print("Desglose de NaN entre las filas perdidas (no son excluyentes, una fila puede tener varios):")
    print(perdidas[["P_ref", "cs_ref", "sp_pred", "potencia_mw"]].isna().sum())
    print()

    # Causa específica: ¿P_ref es NaN porque el t_ref (24h atrás) cae
    # en una franja marcada NaN por la limpieza física (Sección 7)?
    solo_P_ref_nan = perdidas[
        perdidas["P_ref"].isna()
        & perdidas["potencia_mw"].notna()
        & perdidas["cs_ref"].notna()
    ]
    print(f"Filas perdidas SOLO por P_ref NaN (t_ref cayó en hueco/NaN de limpieza física): {len(solo_P_ref_nan):,}")

    solo_potencia_nan = perdidas[
        perdidas["potencia_mw"].isna()
    ]
    print(f"Filas perdidas porque la propia observación (potencia_mw) es NaN: {len(solo_potencia_nan):,}")

    return gp_match, perdidas


# --- Uso ---
claves_wf = df_pred[df_pred["model"] == "WeightedEnsemble"][["item_id","timestamp"]].drop_duplicates()

# claves_wf ya debe existir en el kernel (viene de la celda 46 corregida)
gp_match_yar, perdidas_yar = diagnosticar_planta("YARUCAYA", largo_unificado, df_static, claves_wf)
print()
gp_match_rep, perdidas_rep = diagnosticar_planta("REPARTICION", largo_unificado, df_static, claves_wf)

In [ ]:
# === Construcción de df_taylor y sigma_obs para el Diagrama de Taylor ===
# Agregación: mediana entre plantas (consistente con `resumen`/`resumen_sp`).
# Incluye SmartPersistence como punto de referencia junto a los modelos del
# walk-forward. Usa el mismo universo (item_id, timestamp) por modelo que ya
# se evaluó en `df_res` / `df_sp` -- no se recalcula nada desde cero, solo se
# reutilizan las predicciones ya alineadas.

import numpy as np
import pandas as pd

QUANTILES_TAYLOR = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]  # solo referencia, no se usa aquí

def stats_taylor_por_planta(y_true, y_pred):
    """
    sigma_pred: desviación estándar de la predicción (normalizada por capacidad
                ya viene aplicada aguas arriba, vía nMAE/nRMSE -- aquí usamos
                las series originales en MW y normalizamos al final por planta).
    R:          correlación de Pearson entre observado y predicho.
    cRMSD:      centered RMSD = sqrt(mean((pred - pred.mean()) - (true - true.mean()))^2))
                (la componente de RMSE que NO es bias, tal como exige el diagrama de Taylor).
    """
    sigma_pred = np.std(y_pred, ddof=1)
    R = np.corrcoef(y_true, y_pred)[0, 1]
    anom_true = y_true - y_true.mean()
    anom_pred = y_pred - y_pred.mean()
    crmsd = np.sqrt(np.mean((anom_pred - anom_true) ** 2))
    return sigma_pred, R, crmsd


# --- 1) sigma_obs: desviación estándar de la observación, mediana entre plantas ---
# Se calcula sobre el mismo universo que evaluó WeightedEnsemble (referencia de
# cobertura), usando potencia normalizada por capacidad para que sea comparable
# entre plantas de distinto tamaño -- igual criterio que nMAE/nRMSE.
sigma_obs_por_planta = []
for planta, gp in df[df["model"] == "WeightedEnsemble"].groupby("item_id"):
    cap = CAPACIDADES_MW.get(planta)
    if cap is None:
        continue
    y_true = gp["potencia_mw"].dropna().values
    if len(y_true) < 2:
        continue
    sigma_obs_por_planta.append(np.std(y_true, ddof=1) / cap)

sigma_obs = float(np.median(sigma_obs_por_planta))
print(f"sigma_obs (mediana entre plantas, normalizado por capacidad) = {sigma_obs:.4f}")
print(f"  (calculado sobre {len(sigma_obs_por_planta)} plantas)")


# --- 2) Stats por modelo del walk-forward (AutoGluon), mediana entre plantas ---
filas_taylor = []

for modelo, gm in df.groupby("model"):
    sigmas_planta, Rs_planta, crmsds_planta = [], [], []
    for planta, gp in gm.groupby("item_id"):
        cap = CAPACIDADES_MW.get(planta)
        if cap is None:
            continue
        y_true = gp["potencia_mw"].values
        y_med = gp["0.5"].values
        mask = ~np.isnan(y_true) & ~np.isnan(y_med)
        if mask.sum() < 30:  # mínimo razonable para que std/R no sean ruido
            continue

        yt = y_true[mask] / cap
        ym = y_med[mask] / cap

        s, r, c = stats_taylor_por_planta(yt, ym)
        sigmas_planta.append(s)
        Rs_planta.append(r)
        crmsds_planta.append(c)

    if len(sigmas_planta) == 0:
        continue

    filas_taylor.append({
        "model": modelo,
        "sigma_pred": float(np.median(sigmas_planta)),
        "R": float(np.median(Rs_planta)),
        "cRMSD": float(np.median(crmsds_planta)),
        "n_plantas": len(sigmas_planta),
    })


# --- 3) Stats de SmartPersistence, mismo criterio, reutilizando df_sp ---
# df_sp ya trae nMAE/nRMSE/R por planta (alineado al walk-forward), pero no
# trae sigma_pred ni cRMSD -- los recalculamos aquí con el mismo y_true/y_pred
# que generó df_sp, recorriendo de nuevo el join contra claves_wf.
sigmas_sp, Rs_sp, crmsds_sp = [], [], []

static_idx = df_static.set_index("item_id")
for planta, gp in largo_unificado.groupby("item_id"):
    cap = CAPACIDADES_MW.get(planta)
    if cap is None:
        continue

    lat = static_idx.loc[planta, "latitude"]
    lon = static_idx.loc[planta, "longitude"]
    alt = static_idx.loc[planta, "altitude_m"]

    gp_s = gp.sort_values("timestamp").set_index("timestamp").copy()

    import pvlib
    loc = pvlib.location.Location(lat, lon, altitude=alt)
    cs = loc.get_clearsky(gp_s.index, model="ineichen")
    gp_s["cs_ghi"] = cs["ghi"]
    gp_s["P_ref"] = gp_s["potencia_mw"].shift(24)
    gp_s["cs_ref"] = gp_s["cs_ghi"].shift(24)
    gp_s["sp_pred"] = np.where(
        gp_s["cs_ref"] >= 1.0,
        (gp_s["P_ref"] * gp_s["cs_ghi"] / gp_s["cs_ref"]).clip(lower=0),
        gp_s["P_ref"].clip(lower=0),
    )

    gp_reset = gp_s.reset_index()
    claves_planta = claves_wf[claves_wf["item_id"] == planta][["timestamp"]]
    gp_match = gp_reset.merge(claves_planta, on="timestamp", how="inner")

    mask_eval = (
        ~gp_match["potencia_mw"].isna()
        & ~gp_match["sp_pred"].isna()
        & ~gp_match["P_ref"].isna()
    )
    if mask_eval.sum() < 30:
        continue

    yt = gp_match.loc[mask_eval, "potencia_mw"].values / cap
    yp = gp_match.loc[mask_eval, "sp_pred"].values / cap

    s, r, c = stats_taylor_por_planta(yt, yp)
    sigmas_sp.append(s)
    Rs_sp.append(r)
    crmsds_sp.append(c)

filas_taylor.append({
    "model": "SmartPersistence",
    "sigma_pred": float(np.median(sigmas_sp)),
    "R": float(np.median(Rs_sp)),
    "cRMSD": float(np.median(crmsds_sp)),
    "n_plantas": len(sigmas_sp),
})


# --- 4) Ensamblar df_taylor ---
df_taylor = pd.DataFrame(filas_taylor).sort_values("R", ascending=False).reset_index(drop=True)

print("\n=== df_taylor (mediana entre plantas, normalizado por capacidad) ===")
print(df_taylor.round(4).to_string(index=False))


In [ ]:
# === Diagrama de Taylor con colores por modelo + leyenda con cuadritos ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import skill_metrics as sm

nombres = df_taylor["model"].tolist()
n_modelos = len(nombres)

# Paleta distintiva (10 colores tab10 + 1 extra)
colores = list(plt.cm.tab10(np.linspace(0, 1, 10))) + ["#000000"]
colores = colores[:n_modelos]

# Datos
sigmas = np.array([sigma_obs] + df_taylor["sigma_pred"].tolist())
crmsds = np.array([0.0]       + df_taylor["cRMSD"].tolist())
ccoefs = np.array([1.0]       + df_taylor["R"].tolist())

# Ticks redondeados
sigma_max = max(sigmas) * 1.3
sigma_ticks = np.round(np.linspace(0, sigma_max, 5), 1)
crmsd_max = max(crmsds) * 1.5
crmsd_ticks = np.round(np.linspace(0, crmsd_max, 5), 1)

fig = plt.figure(figsize=(12, 9), facecolor="white")

# Plot base con SkillMetrics (sin labels en los puntos)
sm.taylor_diagram(
    sigmas, crmsds, ccoefs,
    markerLabel=[""] * (n_modelos + 1),   # sin labels — usaremos colores
    markerSize=1,                          # casi invisible — los pintaremos encima
    markerColor="white",
    styleOBS="-",
    colOBS="magenta",
    markerobs="o",
    titleOBS="Observed",
    colCOR="#1f77b4",
    colSTD="black",
    colRMS="green",
    styleCOR="-",
    styleRMS="--",
    widthCOR=0.7,
    widthRMS=0.7,
    showlabelsRMS="on",
    tickRMS=crmsd_ticks,
    titleRMS="on",
    tickSTD=sigma_ticks,
    axismax=sigma_max,
)

# Sobre-pintar cada modelo con color distintivo
ax = plt.gca()
for i, (nombre, color) in enumerate(zip(nombres, colores)):
    R = df_taylor.iloc[i]["R"]
    s = df_taylor.iloc[i]["sigma_pred"]
    theta = np.arccos(R)
    x = s * np.cos(theta)
    y = s * np.sin(theta)
    ax.plot(x, y, marker="o", markersize=11, color=color,
            markeredgecolor="black", markeredgewidth=0.6, zorder=10,
            linestyle="none")

# Título limpio (movido arriba)
plt.suptitle(
    "Diagrama de Taylor — Pronóstico Solar PV Day-Ahead",
    fontsize=12, y=0.97
)
plt.title(
    "test_split 24h | 11 plantas SEIN-Perú",
    fontsize=10, pad=10
)

# Leyenda externa con cuadritos de color
legend_elements = [
    Line2D([0], [0], marker="o", color="w",
           markerfacecolor=color, markeredgecolor="black",
           markersize=10, label=nombre)
    for nombre, color in zip(nombres, colores)
]
fig.legend(
    handles=legend_elements,
    loc="center right",
    bbox_to_anchor=(0.98, 0.5),
    fontsize=10,
    frameon=True,
    edgecolor="gray",
    framealpha=0.95,
    title="Modelos",
    title_fontsize=11,
)

plt.subplots_adjust(right=0.72)

# Crear el directorio si no existe -- plt.savefig NO lo crea automáticamente
from pathlib import Path
OUT_DIR_TAYLOR = Path("models/autogluon_v1")
OUT_DIR_TAYLOR.mkdir(parents=True, exist_ok=True)

plt.savefig(OUT_DIR_TAYLOR / "diagrama_taylor.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT_DIR_TAYLOR / "diagrama_taylor.pdf", bbox_inches="tight")
plt.show()
print(f"✓ Diagrama guardado")

In [ ]:
# === Sanity check: coordenadas de cada punto en el Taylor ===
print(f"σ_obs = {sigma_obs:.3f}")
print(f"\n{'Modelo':<35} {'R':>8} {'σ_pred':>10} {'cRMSD':>10}")
print("-" * 65)
for _, row in df_taylor.iterrows():
    print(f"{row['model']:<35} {row['R']:>8.4f} {row['sigma_pred']:>10.3f} {row['cRMSD']:>10.3f}")

# Detectar pares solapados
print(f"\n=== Posibles solapamientos (Δσ<0.5 y ΔR<0.005) ===")
for i in range(len(df_taylor)):
    for j in range(i+1, len(df_taylor)):
        a, b = df_taylor.iloc[i], df_taylor.iloc[j]
        d_sigma = abs(a["sigma_pred"] - b["sigma_pred"])
        d_R = abs(a["R"] - b["R"])
        if d_sigma < 0.5 and d_R < 0.005:
            print(f"  {a['model']:<30} y {b['model']:<30}: Δσ={d_sigma:.3f}, ΔR={d_R:.4f}")

In [ ]:
# === Diagramas de Taylor por planta (uno por estación) ===
# Reutiliza stats_taylor_por_planta() de la celda anterior. En vez de agregar
# por mediana entre plantas, aquí se calcula sigma/R/cRMSD POR PLANTA para
# cada modelo, y se genera un diagrama de Taylor independiente por planta.
# Útil para auditar si la mediana global está ocultando comportamientos muy
# distintos entre estaciones (ej. una planta donde el ensemble pierde feo
# contra SmartPersistence aunque en la mediana gane).

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path
import skill_metrics as sm

OUT_DIR_TAYLOR_PLANTA = Path("models/autogluon_v1/taylor_por_planta")
OUT_DIR_TAYLOR_PLANTA.mkdir(parents=True, exist_ok=True)

# Mismo umbral mínimo de muestra que se usó al construir df_taylor agregado
MIN_N_PLANTA = 30

plantas_disponibles = sorted(df["item_id"].unique())
print(f"Generando diagramas de Taylor para {len(plantas_disponibles)} plantas...\n")

# Paleta fija y consistente entre TODAS las plantas, para poder comparar
# visualmente el mismo color = mismo modelo a través de los 11 gráficos
todos_los_modelos = sorted(df["model"].unique().tolist()) + ["SmartPersistence"]
n_total_modelos = len(todos_los_modelos)
# Generar exactamente un color por modelo -- no asumir un tope fijo de 10.
# Si hay <=10 modelos, usa tab10 (colores más distintivos). Si hay más,
# usa tab20 y, si aun así no alcanza, completa con hsv.
if n_total_modelos <= 10:
    paleta_colores = list(plt.cm.tab10(np.linspace(0, 1, n_total_modelos)))
elif n_total_modelos <= 20:
    paleta_colores = list(plt.cm.tab20(np.linspace(0, 1, n_total_modelos)))
else:
    paleta_colores = list(plt.cm.hsv(np.linspace(0, 0.95, n_total_modelos)))

paleta_global = dict(zip(todos_los_modelos, paleta_colores))
print(f"Paleta generada para {n_total_modelos} modelos: {todos_los_modelos}")

resumen_por_planta = []  # para inspección tabular antes/después de graficar

for planta in plantas_disponibles:
    cap = CAPACIDADES_MW.get(planta)
    if cap is None:
        print(f"  ⚠ {planta}: sin capacidad nominal, se omite")
        continue

    # --- sigma_obs de ESTA planta (no la mediana global) ---
    gp_obs = df[(df["model"] == "WeightedEnsemble") & (df["item_id"] == planta)]
    y_obs = gp_obs["potencia_mw"].dropna().values / cap
    if len(y_obs) < MIN_N_PLANTA:
        print(f"  ⚠ {planta}: solo {len(y_obs)} obs válidas (<{MIN_N_PLANTA}), se omite")
        continue
    sigma_obs_planta = np.std(y_obs, ddof=1)

    # --- stats por modelo del walk-forward, para ESTA planta ---
    filas_planta = []
    for modelo, gm in df[df["item_id"] == planta].groupby("model"):
        y_true = gm["potencia_mw"].values
        y_med = gm["0.5"].values
        mask = ~np.isnan(y_true) & ~np.isnan(y_med)
        if mask.sum() < MIN_N_PLANTA:
            continue
        yt = y_true[mask] / cap
        ym = y_med[mask] / cap
        s, r, c = stats_taylor_por_planta(yt, ym)
        filas_planta.append({"model": modelo, "sigma_pred": s, "R": r, "cRMSD": c, "n": mask.sum()})

    # --- SmartPersistence para ESTA planta, mismo join que ya validamos ---
    static_idx = df_static.set_index("item_id")
    gp = largo_unificado[largo_unificado.item_id == planta].sort_values("timestamp").set_index("timestamp").copy()
    lat = static_idx.loc[planta, "latitude"]
    lon = static_idx.loc[planta, "longitude"]
    alt = static_idx.loc[planta, "altitude_m"]

    import pvlib
    loc = pvlib.location.Location(lat, lon, altitude=alt)
    cs = loc.get_clearsky(gp.index, model="ineichen")
    gp["cs_ghi"] = cs["ghi"]
    gp["P_ref"] = gp["potencia_mw"].shift(24)
    gp["cs_ref"] = gp["cs_ghi"].shift(24)
    gp["sp_pred"] = np.where(
        gp["cs_ref"] >= 1.0,
        (gp["P_ref"] * gp["cs_ghi"] / gp["cs_ref"]).clip(lower=0),
        gp["P_ref"].clip(lower=0),
    )
    gp_reset = gp.reset_index()
    claves_planta = claves_wf[claves_wf["item_id"] == planta][["timestamp"]]
    gp_match = gp_reset.merge(claves_planta, on="timestamp", how="inner")
    mask_sp = ~gp_match["potencia_mw"].isna() & ~gp_match["sp_pred"].isna() & ~gp_match["P_ref"].isna()

    if mask_sp.sum() >= MIN_N_PLANTA:
        yt_sp = gp_match.loc[mask_sp, "potencia_mw"].values / cap
        yp_sp = gp_match.loc[mask_sp, "sp_pred"].values / cap
        s, r, c = stats_taylor_por_planta(yt_sp, yp_sp)
        filas_planta.append({"model": "SmartPersistence", "sigma_pred": s, "R": r, "cRMSD": c, "n": mask_sp.sum()})

    if len(filas_planta) == 0:
        print(f"  ⚠ {planta}: sin modelos con muestra suficiente, se omite")
        continue

    df_taylor_planta = pd.DataFrame(filas_planta).sort_values("R", ascending=False).reset_index(drop=True)
    for _, row in df_taylor_planta.iterrows():
        resumen_por_planta.append({"item_id": planta, **row.to_dict()})

    # --- Graficar ---
    nombres = df_taylor_planta["model"].tolist()
    colores = [paleta_global[n] for n in nombres]

    sigmas = np.array([sigma_obs_planta] + df_taylor_planta["sigma_pred"].tolist())
    crmsds = np.array([0.0] + df_taylor_planta["cRMSD"].tolist())
    ccoefs = np.array([1.0] + df_taylor_planta["R"].tolist())

    sigma_max = max(sigmas) * 1.3
    sigma_ticks = np.round(np.linspace(0, sigma_max, 5), 2)
    crmsd_max = max(crmsds) * 1.5
    crmsd_ticks = np.round(np.linspace(0, crmsd_max, 5), 2)

    fig = plt.figure(figsize=(10, 8), facecolor="white")

    sm.taylor_diagram(
        sigmas, crmsds, ccoefs,
        markerLabel=[""] * (len(nombres) + 1),
        markerSize=1,
        markerColor="white",
        styleOBS="-", colOBS="magenta", markerobs="o", titleOBS="Observed",
        colCOR="#1f77b4", colSTD="black", colRMS="green",
        styleCOR="-", styleRMS="--", widthCOR=0.7, widthRMS=0.7,
        showlabelsRMS="on", tickRMS=crmsd_ticks, titleRMS="on",
        tickSTD=sigma_ticks, axismax=sigma_max,
    )

    ax = plt.gca()
    for i, (nombre, color) in enumerate(zip(nombres, colores)):
        R = df_taylor_planta.iloc[i]["R"]
        s = df_taylor_planta.iloc[i]["sigma_pred"]
        theta = np.arccos(np.clip(R, -1, 1))
        x = s * np.cos(theta)
        y = s * np.sin(theta)
        ax.plot(x, y, marker="o", markersize=11, color=color,
                markeredgecolor="black", markeredgewidth=0.6, zorder=10, linestyle="none")

    plt.suptitle(f"Diagrama de Taylor — {planta}", fontsize=12, y=0.97)
    plt.title(f"Pronóstico Solar PV Day-Ahead | n={df_taylor_planta['n'].max()}h evaluadas", fontsize=10, pad=10)

    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=color,
               markeredgecolor="black", markersize=10, label=nombre)
        for nombre, color in zip(nombres, colores)
    ]
    fig.legend(handles=legend_elements, loc="center right", bbox_to_anchor=(0.98, 0.5),
               fontsize=9, frameon=True, edgecolor="gray", framealpha=0.95,
               title="Modelos", title_fontsize=10)

    plt.subplots_adjust(right=0.72)
    out_png = OUT_DIR_TAYLOR_PLANTA / f"taylor_{planta}.png"
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"  ✓ {planta}: guardado en {out_png}")

df_taylor_por_planta = pd.DataFrame(resumen_por_planta)
print(f"\n✓ {df_taylor_por_planta['item_id'].nunique()} diagramas generados en {OUT_DIR_TAYLOR_PLANTA}/")
print("\n=== Tabla completa: stats por planta y modelo ===")
print(df_taylor_por_planta.round(4).to_string(index=False))


In [ ]:
# === Permutation Importance — por modelo, subplots ===
# Se calcula PI para cada modelo que usa covariables.
# Se presentan en subplots separados para comparar sin mezclar escalas.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

FIT_DIR = Path("models/walkforward_v1/fits")
OUT_DIR = Path("models/walkforward_v1/analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

METEO        = ["ghi", "t_air", "hr", "wind_speed", "pressure", "precip_mm"]
METEO_LABELS = {"ghi": "GHI", "t_air": "T_aire", "hr": "HR",
                "wind_speed": "Wind speed", "pressure": "Pressure",
                "precip_mm": "Precip"}
EXCLUIR = {"SeasonalNaive", "DynamicOptimizedTheta", "AutoETS"}

# ── 1. Cargar predictor ──────────────────────────────────────────────────────
ultimo_fit   = sorted(FIT_DIR.glob("fit_*/predictor.pkl"))[-1].parent
predictor_fi = TimeSeriesPredictor.load(str(ultimo_fit))
print(f"Predictor cargado: {ultimo_fit.name}\n")

# ── 2. Dataset ───────────────────────────────────────────────────────────────
cols_ts = ["item_id", "timestamp", "potencia_mw"] + METEO
test_fi = TimeSeriesDataFrame.from_data_frame(
    largo_unificado[cols_ts].dropna().copy(),
    id_column="item_id",
    timestamp_column="timestamp",
    static_features_df=df_static,
)

# ── 3. PI por modelo ─────────────────────────────────────────────────────────
importancias = {}
for modelo in predictor_fi.model_names():
    if modelo in EXCLUIR:
        print(f"— {modelo}: excluido")
        continue
    try:
        imp = predictor_fi.feature_importance(
            data=test_fi,
            model=modelo,
            features=METEO,
            subsample_size=500,
        )
        imp_meteo = (
            imp["importance"]
            .reindex([f for f in METEO if f in imp.index])
            .dropna()
            .rename(METEO_LABELS)
        )
        if imp_meteo.empty:
            print(f"✗ {modelo}: sin met vars")
            continue
        importancias[modelo] = imp_meteo
        print(f"✓ {modelo}  GHI={imp_meteo.get('GHI', 0):.4f}")
    except Exception as e:
        print(f"✗ {modelo}: {e}")

# ── 4. Subplots — uno por modelo ─────────────────────────────────────────────
modelos = list(importancias.keys())
n       = len(modelos)
ncols   = 3
nrows   = -(-n // ncols)   # ceil

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(5.5 * ncols, 4.0 * nrows),
                          facecolor="white")
axes = axes.flatten()

for i, modelo in enumerate(modelos):
    ax      = axes[i]
    imp_m   = importancias[modelo]
    orden   = imp_m.sort_values(ascending=True).index
    imp_ord = imp_m.loc[orden]
    colores = ["#2c7fb8" if v >= 0 else "#d7301f" for v in imp_ord.values]

    ax.barh(imp_ord.index, imp_ord.values,
            color=colores, edgecolor="black", linewidth=0.5)
    ax.axvline(0, color="black", linewidth=0.7)
    ax.set_title(modelo, fontsize=10, pad=6)
    ax.set_xlabel("Δ SQL", fontsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.tick_params(axis="x", labelsize=7)

# Ocultar ejes sobrantes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    "Permutation Importance por modelo — Covariables meteorológicas\n"
    "11 plantas SEIN-Perú · Walk-forward",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig(OUT_DIR / "perm_importance_por_modelo.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT_DIR / "perm_importance_por_modelo.pdf", bbox_inches="tight")
plt.show()

# ── 5. Persistir tabla completa ──────────────────────────────────────────────
df_imp = pd.DataFrame(importancias).T.reindex(
    columns=[METEO_LABELS[c] for c in METEO]
)
df_imp.to_csv(OUT_DIR / "perm_importance_por_modelo.csv")
print(f"\n=== Tabla completa ===")
print(df_imp.to_string())
print(f"\n✓ Guardado en {OUT_DIR}")

In [ ]:
from pathlib import Path
from autogluon.timeseries import TimeSeriesPredictor

FIT_DIR    = Path("models/walkforward_v1/fits")
ultimo_fit = sorted(FIT_DIR.glob("fit_*/predictor.pkl"))[-1].parent
p          = TimeSeriesPredictor.load(str(ultimo_fit))

dt = p._learner.load_trainer().load_model("DirectTabular")
print(type(dt))

# Bajar al modelo interno
for attr in ["model", "models", "_model", "most_recent_model", "base_models"]:
    if hasattr(dt, attr):
        obj = getattr(dt, attr)
        print(f"dt.{attr} -> {type(obj)}")

In [ ]:
dt2 = dt.most_recent_model
print([a for a in dir(dt2) if not a.startswith('__')])

In [ ]:
dt2 = dt.most_recent_model
print(type(dt2.covariate_regressor))
print(type(dt2.get_tabular_model()))

In [ ]:
tm = dt.most_recent_model.get_tabular_model()
lgb_q50 = tm.model.model.model_dict[0.5]
print(lgb_q50.feature_name()[:8])

In [ ]:
# === SHAP sobre DirectTabular (q=0.5), solo las 6 covariables meteorológicas ===
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
from autogluon.timeseries import TimeSeriesPredictor

FIT_DIR  = Path("models/walkforward_v1/fits")
OUT_DIR  = Path("models/autogluon_v1")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Extraer booster LightGBM q=0.5 de DirectTabular ──────────────────────
ultimo_fit = sorted(FIT_DIR.glob("fit_*/predictor.pkl"))[-1].parent
predictor_shap = TimeSeriesPredictor.load(str(ultimo_fit))

dt      = predictor_shap._learner.load_trainer().load_model("DirectTabular")
tm      = dt.most_recent_model.get_tabular_model()
lgb_q50 = tm.model.model.model_dict[0.5]   # MultiWindowBacktestingModel -> DirectTabularModel -> TabularModel -> LGBModel -> QuantileBooster -> booster q=0.5
feat_names = lgb_q50.feature_name()
print(f"Features del booster q=0.5: {len(feat_names)}")

# ── 2. Reconstruir feature matrix ────────────────────────────────────────────
df = largo_unificado.dropna(subset=["potencia_mw"]).copy()
df = df.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

LAGS = [1,2,3,4,5,6,7,23,24,25,47,48,49,71,72,73,95,96,97,
        119,120,121,143,144,145,167,168,169,
        335,336,337,503,504,505,671,672,673,719,720,721]
for lag in LAGS:
    df[f"lag{lag}"] = df.groupby("item_id")["potencia_mw"].shift(lag)

for c in ["ghi","t_air","hr","wind_speed","pressure","precip_mm"]:
    df[f"__scaled_{c}"] = df.groupby("item_id")[c].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8)
    )

ts = pd.to_datetime(df["timestamp"])
df["hour_of_day"]  = ts.dt.hour
df["day_of_week"]  = ts.dt.dayofweek
df["day_of_month"] = ts.dt.day
df["day_of_year"]  = ts.dt.dayofyear

df = df.merge(df_static, on="item_id", how="left")

cols = [c for c in feat_names if c in df.columns]
df_clean = df.dropna(subset=cols)
X = df_clean[cols].copy()

for c in ["tracking","modulo_fabricante","modulo_tecnologia","inversor_fabricante"]:
    if c in X.columns:
        X[c] = X[c].astype("category")

print(f"X shape: {X.shape}")

# ── 3. SHAP TreeExplainer ────────────────────────────────────────────────────
X_sample    = X.sample(n=10000, random_state=42)
explainer   = shap.TreeExplainer(lgb_q50)
shap_values = explainer.shap_values(X_sample)
print(f"SHAP values shape: {shap_values.shape}")

# ── 4. Filtrar las 6 covariables para el plot ─────────────────────────────────
covariables_6  = ["__scaled_ghi","__scaled_t_air","__scaled_hr",
                  "__scaled_wind_speed","__scaled_pressure","__scaled_precip_mm"]
nombres_limpios = ["GHI","T_aire","HR","Wind speed","Pressure","Precip"]

idx_6   = [list(X_sample.columns).index(c) for c in covariables_6 if c in X_sample.columns]
shap_6  = shap_values[:, idx_6]
X_6     = X_sample.iloc[:, idx_6].copy()
X_6.columns = nombres_limpios[:len(idx_6)]

print(f"Features en el plot: {X_6.columns.tolist()}")

# ── 5. Beeswarm ──────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 6), facecolor="white")
shap.summary_plot(shap_6, X_6, plot_type="dot", show=False, max_display=6)
plt.title(
    "SHAP — Covariables meteorológicas en DirectTabular (q=0.5)\n"
    "11 plantas SEIN-Perú | Modelo del leaderboard",
    fontsize=11, pad=12
)
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_beeswarm.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT_DIR / "shap_beeswarm.pdf", bbox_inches="tight")
plt.show()

# ── 6. Bar ───────────────────────────────────────────────────────────────────
plt.figure(figsize=(10, 5), facecolor="white")
shap.summary_plot(shap_6, X_6, plot_type="bar", show=False, max_display=6)
plt.title("SHAP Mean |Value| — 6 covariables", fontsize=11, pad=12)
plt.tight_layout()
plt.savefig(OUT_DIR / "shap_bar.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT_DIR / "shap_bar.pdf", bbox_inches="tight")
plt.show()

# ── 7. Persistir ─────────────────────────────────────────────────────────────
with open(OUT_DIR / "shap_results.pkl", "wb") as f:
    pickle.dump({
        "shap_values_completo": shap_values,
        "shap_values_6cov": shap_6,
        "X_sample": X_sample,
        "X_6": X_6,
        "feature_names_completo": cols,
        "feature_names_6cov": nombres_limpios,
    }, f)

print(f"\n✓ SHAP DirectTabular completo — guardado en {OUT_DIR}")

In [ ]:
# === Predicción de un día completo — subplots por modelo ===

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

PRED_DIR = Path("models/walkforward_v1/preds")
OUT_DIR  = Path("models/walkforward_v1/analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Cargar predicciones ───────────────────────────────────────────────────
archivos = sorted(PRED_DIR.glob("*.parquet"))
df = pd.concat([pd.read_parquet(f) for f in archivos], ignore_index=True)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["dia"] = df["timestamp"].dt.normalize()

# ── 2. Observado real desde largo_unificado ──────────────────────────────────
obs_real = (
    largo_unificado[["item_id", "timestamp", "potencia_mw"]]
    .copy()
    .assign(timestamp=lambda x: pd.to_datetime(x["timestamp"]))
    .drop_duplicates(subset=["item_id", "timestamp"])
)

# ── 3. Elegir planta y día representativo ────────────────────────────────────
PLANTA = "CLEMESI"
CAP_MW = df_static.set_index("item_id").loc[PLANTA, "pot_inst_mw"]

# Día con mayor generación real observada:
# 1. Solo días con predicciones completas (>=20 registros en WE)
# 2. Solo días con observado completo (>=20 filas no nulas)
# 3. Mayor potencia media observada entre esos días
obs_planta = obs_real[obs_real["item_id"] == PLANTA].copy()
obs_planta["dia"] = obs_planta["timestamp"].dt.normalize()

df_we = df[(df["item_id"] == PLANTA) & (df["model"] == "WeightedEnsemble")].copy()
df_we["dia"] = df_we["timestamp"].dt.normalize()

# Días con predicción completa (al menos 20 registros horarios)
dias_pred_ok = df_we.groupby("dia").size()
dias_pred_ok = dias_pred_ok[dias_pred_ok >= 20].index

# Días con observado completo y no nulo
dias_obs_ok = (
    obs_planta[obs_planta["dia"].isin(dias_pred_ok)]
    .groupby("dia")["potencia_mw"]
    .apply(lambda x: x.notna().sum())
)
dias_obs_ok = dias_obs_ok[dias_obs_ok >= 20].index

# Día con mayor potencia media observada entre los válidos
dia_max = (
    obs_planta[obs_planta["dia"].isin(dias_obs_ok)]
    .groupby("dia")["potencia_mw"]
    .mean()
    .sort_values(ascending=False)
    .index[0]
)
print(f"Día seleccionado: {dia_max.date()} | Planta: {PLANTA} | Cap: {CAP_MW} MW")

# Observado de ese día — resampleado a horario para alinear con la predicción (24h)
obs_dia_raw = obs_planta[obs_planta["dia"] == dia_max].sort_values("timestamp")
obs_dia = (
    obs_dia_raw.set_index("timestamp")["potencia_mw"]
    .resample("h")
    .mean()
    .reset_index()
)
print(f"Filas observadas (horario): {len(obs_dia)} | Max potencia: {obs_dia['potencia_mw'].max():.1f} MW")

# ── 4. Predicciones de ese día ───────────────────────────────────────────────
df_dia = df[(df["item_id"] == PLANTA) & (df["dia"] == dia_max)].copy()
df_dia = df_dia.sort_values("timestamp")

modelos_orden = [
    "WeightedEnsemble", "DirectTabular", "Chronos2",
    "Chronos2SmallFineTuned", "ChronosWithRegressor[bolt_small]",
    "TemporalFusionTransformer", "DeepAR", "RecursiveTabular",
    "SeasonalNaive", "AutoETS", "DynamicOptimizedTheta",
]
modelos_presentes = [m for m in modelos_orden if m in df_dia["model"].unique()]
n = len(modelos_presentes)

# ── 5. Subplots ──────────────────────────────────────────────────────────────
ncols = 3
nrows = -(-n // ncols)
fig, axes = plt.subplots(nrows, ncols,
                          figsize=(6 * ncols, 3.8 * nrows),
                          facecolor="white", sharey=True)
axes = axes.flatten()

COLORES = {
    "WeightedEnsemble":                 "#1a6faf",
    "DirectTabular":                    "#e05c00",
    "Chronos2":                         "#2ca02c",
    "Chronos2SmallFineTuned":           "#9467bd",
    "ChronosWithRegressor[bolt_small]": "#8c564b",
    "TemporalFusionTransformer":        "#e377c2",
    "DeepAR":                           "#17becf",
    "RecursiveTabular":                 "#bcbd22",
    "SeasonalNaive":                    "#7f7f7f",
    "AutoETS":                          "#aec7e8",
    "DynamicOptimizedTheta":            "#ffbb78",
}

for i, modelo in enumerate(modelos_presentes):
    ax  = axes[i]
    dm  = df_dia[df_dia["model"] == modelo].sort_values("timestamp")
    col = COLORES.get(modelo, "#333333")

    # Banda cuantílica [q=0.1, q=0.9] — clipeada a [0, CAP_MW]
    q10 = dm["0.1"].clip(0, CAP_MW)
    q90 = dm["0.9"].clip(0, CAP_MW)
    ax.fill_between(dm["timestamp"], q10, q90,
                    alpha=0.25, color=col, label="[q0.1, q0.9]")

    # Mediana predicha
    ax.plot(dm["timestamp"], dm["0.5"].clip(0, CAP_MW),
            color=col, linewidth=1.8, label="Mediana")

    # Observado REAL desde largo_unificado
    ax.plot(obs_dia["timestamp"], obs_dia["potencia_mw"],
            color="black", linewidth=1.4, linestyle="--", label="Observado")

    ax.set_title(modelo, fontsize=9.5, pad=5)
    ax.set_ylabel("MW", fontsize=8)
    ax.set_ylim(-5, CAP_MW * 1.05)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=4))
    ax.tick_params(axis="x", labelsize=7, rotation=30)
    ax.tick_params(axis="y", labelsize=7)
    ax.grid(True, alpha=0.3, linewidth=0.5)

    if i == 0:
        ax.legend(fontsize=7, loc="upper left")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    f"Pronóstico day-ahead — {PLANTA} · {dia_max.date()}\n"
    "Mediana predicha + banda [q0.1, q0.9] vs. observado real",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig(OUT_DIR / "prediccion_dia_completo.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT_DIR / "prediccion_dia_completo.pdf", bbox_inches="tight")
plt.show()
print(f"✓ Guardado en {OUT_DIR}")

In [ ]:
# === 1. Reliability diagram (calibración cuantílica) ===
# Para cada cuantil nominal q, mide qué fracción de observaciones
# cae por debajo de la predicción. Un modelo bien calibrado tiene
# cobertura empírica ≈ cobertura nominal (diagonal perfecta).

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PRED_DIR = Path("models/walkforward_v1/preds")
OUT_DIR  = Path("models/walkforward_v1/analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

archivos = sorted(PRED_DIR.glob("*.parquet"))
df = pd.concat([pd.read_parquet(f) for f in archivos], ignore_index=True)
df["timestamp"] = pd.to_datetime(df["timestamp"])

QUANTILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

MODELOS_ORDEN = [
    "WeightedEnsemble", "DirectTabular", "Chronos2",
    "Chronos2SmallFineTuned", "ChronosWithRegressor[bolt_small]",
    "TemporalFusionTransformer", "DeepAR", "RecursiveTabular",
    "SeasonalNaive", "AutoETS", "DynamicOptimizedTheta",
]
COLORES = {
    "WeightedEnsemble":                 "#1a6faf",
    "DirectTabular":                    "#e05c00",
    "Chronos2":                         "#2ca02c",
    "Chronos2SmallFineTuned":           "#9467bd",
    "ChronosWithRegressor[bolt_small]": "#8c564b",
    "TemporalFusionTransformer":        "#e377c2",
    "DeepAR":                           "#17becf",
    "RecursiveTabular":                 "#bcbd22",
    "SeasonalNaive":                    "#7f7f7f",
    "AutoETS":                          "#aec7e8",
    "DynamicOptimizedTheta":            "#ffbb78",
}

fig, ax = plt.subplots(figsize=(7, 6), facecolor="white")
ax.plot([0, 1], [0, 1], "k--", linewidth=1.2, label="Calibración perfecta", zorder=10)

modelos_presentes = [m for m in MODELOS_ORDEN if m in df["model"].unique()]
for modelo in modelos_presentes:
    dm = df[df["model"] == modelo].dropna(subset=["potencia_mw"])
    coberturas = []
    for q in QUANTILES:
        yq  = dm[str(q)].values
        yt  = dm["potencia_mw"].values
        mask = ~np.isnan(yq) & ~np.isnan(yt)
        coberturas.append((yt[mask] <= yq[mask]).mean())

    lw  = 2.5 if modelo == "WeightedEnsemble" else 1.2
    zord = 9 if modelo == "WeightedEnsemble" else 5
    ax.plot(QUANTILES, coberturas,
            color=COLORES.get(modelo, "#333"),
            linewidth=lw, marker="o", markersize=4,
            label=modelo, zorder=zord)

ax.set_xlabel("Cuantil nominal", fontsize=11)
ax.set_ylabel("Cobertura empírica", fontsize=11)
ax.set_title(
    "Reliability Diagram — Calibración cuantílica\n"
    "11 plantas SEIN-Perú · Walk-forward",
    fontsize=11, pad=10
)
ax.set_xlim(0.05, 0.95)
ax.set_ylim(0.05, 0.95)
ax.legend(fontsize=7.5, loc="upper left", ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "calibracion_cuantilica.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT_DIR / "calibracion_cuantilica.pdf", bbox_inches="tight")
plt.show()
print(f"✓ Guardado en {OUT_DIR}")

In [ ]:
# === 2. Error por hora del día (nMAE) — todos los modelos ===
# Muestra en qué tramo del día falla más cada modelo.
# WE destacado con línea gruesa.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

PRED_DIR = Path("models/walkforward_v1/preds")
OUT_DIR  = Path("models/walkforward_v1/analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

archivos = sorted(PRED_DIR.glob("*.parquet"))
df = pd.concat([pd.read_parquet(f) for f in archivos], ignore_index=True)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df["hora"] = df["timestamp"].dt.hour

CAPACIDADES = df_static.set_index("item_id")["pot_inst_mw"].to_dict()
MODELOS_ORDEN = [
    "WeightedEnsemble", "DirectTabular", "Chronos2",
    "Chronos2SmallFineTuned", "ChronosWithRegressor[bolt_small]",
    "TemporalFusionTransformer", "DeepAR", "RecursiveTabular",
    "SeasonalNaive", "AutoETS", "DynamicOptimizedTheta",
]
COLORES = {
    "WeightedEnsemble":                 "#1a6faf",
    "DirectTabular":                    "#e05c00",
    "Chronos2":                         "#2ca02c",
    "Chronos2SmallFineTuned":           "#9467bd",
    "ChronosWithRegressor[bolt_small]": "#8c564b",
    "TemporalFusionTransformer":        "#e377c2",
    "DeepAR":                           "#17becf",
    "RecursiveTabular":                 "#bcbd22",
    "SeasonalNaive":                    "#7f7f7f",
    "AutoETS":                          "#aec7e8",
    "DynamicOptimizedTheta":            "#ffbb78",
}

fig, ax = plt.subplots(figsize=(10, 5), facecolor="white")

modelos_presentes = [m for m in MODELOS_ORDEN if m in df["model"].unique()]
for modelo in modelos_presentes:
    dm = df[df["model"] == modelo].copy()
    dm["cap"] = dm["item_id"].map(CAPACIDADES)
    dm["ae_norm"] = (dm["potencia_mw"] - dm["0.5"]).abs() / dm["cap"]
    nmae_hora = dm.groupby("hora")["ae_norm"].mean()

    lw   = 2.8 if modelo == "WeightedEnsemble" else 1.0
    alpha = 1.0 if modelo == "WeightedEnsemble" else 0.65
    zord  = 9  if modelo == "WeightedEnsemble" else 4
    ax.plot(nmae_hora.index, nmae_hora.values,
            color=COLORES.get(modelo, "#333"),
            linewidth=lw, alpha=alpha,
            label=modelo, zorder=zord)

ax.set_xlabel("Hora del día (UTC-5)", fontsize=11)
ax.set_ylabel("nMAE medio", fontsize=11)
ax.set_xticks(range(0, 24, 2))
ax.set_title(
    "Error por hora del día — nMAE medio por modelo\n"
    "11 plantas SEIN-Perú · Walk-forward",
    fontsize=11, pad=10
)
ax.legend(fontsize=7.5, loc="upper right", ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "error_por_hora.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT_DIR / "error_por_hora.pdf", bbox_inches="tight")
plt.show()
print(f"✓ Guardado en {OUT_DIR}")

In [ ]:
# === 3. Heatmap nMAE — modelos × plantas ===
# Revela si algún modelo domina en todas las plantas o hay heterogeneidad
# geográfica (costa norte vs. sur andino vs. altiplano).

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PRED_DIR = Path("models/walkforward_v1/preds")
OUT_DIR  = Path("models/walkforward_v1/analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

archivos = sorted(PRED_DIR.glob("*.parquet"))
df = pd.concat([pd.read_parquet(f) for f in archivos], ignore_index=True)
CAPACIDADES = df_static.set_index("item_id")["pot_inst_mw"].to_dict()

MODELOS_ORDEN = [
    "WeightedEnsemble", "DirectTabular", "Chronos2",
    "Chronos2SmallFineTuned", "ChronosWithRegressor[bolt_small]",
    "TemporalFusionTransformer", "DeepAR", "RecursiveTabular",
    "SeasonalNaive", "AutoETS", "DynamicOptimizedTheta",
]

resultados = []
for modelo, gm in df.groupby("model"):
    for planta, gp in gm.groupby("item_id"):
        cap = CAPACIDADES.get(planta)
        if cap is None:
            continue
        mask = gp["potencia_mw"].notna() & gp["0.5"].notna()
        if mask.sum() == 0:
            continue
        nmae = (gp.loc[mask, "potencia_mw"] - gp.loc[mask, "0.5"]).abs().mean() / cap
        resultados.append({"model": modelo, "item_id": planta, "nMAE": nmae})

df_res = pd.DataFrame(resultados)
pivot = df_res.pivot(index="model", columns="item_id", values="nMAE")
pivot = pivot.reindex([m for m in MODELOS_ORDEN if m in pivot.index])

fig, ax = plt.subplots(
    figsize=(max(7, 0.8 * pivot.shape[1] + 2), max(4, 0.5 * pivot.shape[0] + 1.5)),
    facecolor="white"
)
sns.heatmap(
    pivot, cmap="YlOrRd", annot=True, fmt=".3f",
    annot_kws={"size": 8}, linewidths=0.4, linecolor="white",
    cbar_kws={"label": "nMAE"}, ax=ax
)
ax.set_title(
    "nMAE por modelo y planta — Walk-forward\n"
    "11 plantas SEIN-Perú",
    fontsize=11, pad=10
)
plt.setp(ax.get_xticklabels(), rotation=35, ha="right", fontsize=8)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_nmae_modelos_plantas.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT_DIR / "heatmap_nmae_modelos_plantas.pdf", bbox_inches="tight")
plt.show()
print(f"✓ Guardado en {OUT_DIR}")

In [ ]:
from pathlib import Path
p = Path("data/raw/era5_tcc/RUBI_tcc.nc")
print(p.stat().st_size)
with open(p, "rb") as f:
    print(f.read(8).hex())

In [ ]:
# === nMAE por régimen de nubosidad (TCC ERA5) — heatmap + barras ===

import os
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PRED_DIR    = Path("models/walkforward_v1/preds")
ERA5_TCC_DIR = Path("data/raw/era5_tcc")
OUT_DIR     = Path("models/walkforward_v1/analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Umbrales TCC (fracción 0–1)
BINS   = [-0.01, 0.2, 0.6, 1.01]
LABELS = ["Despejado", "Parc. Nublado", "Muy Nublado"]

MODELOS_ORDEN = [
    "WeightedEnsemble", "DirectTabular", "Chronos2",
    "Chronos2SmallFineTuned", "ChronosWithRegressor[bolt_small]",
    "TemporalFusionTransformer", "DeepAR", "RecursiveTabular",
    "SeasonalNaive", "AutoETS", "DynamicOptimizedTheta",
]

# ── 1. Cargar TCC de los 11 archivos .nc (CDS los baja como ZIP) ────────────
import zipfile, tempfile, shutil

tcc_list = []
for nc_path in sorted(ERA5_TCC_DIR.glob("*_tcc.nc")):
    item_id = nc_path.stem.replace("_tcc", "")

    # Detectar si es ZIP y descomprimir en un dir temporal
    with open(nc_path, "rb") as f:
        magic = f.read(4)
    is_zip = magic == b"PK\x03\x04"

    if is_zip:
        tmp_dir = Path(tempfile.mkdtemp())
        with zipfile.ZipFile(nc_path, "r") as zf:
            zf.extractall(tmp_dir)
        # El .nc real está dentro del ZIP
        inner = sorted(tmp_dir.glob("*.nc"))
        if not inner:
            inner = sorted(tmp_dir.rglob("*"))
        real_path = inner[0]
    else:
        real_path = nc_path
        tmp_dir   = None

    # Abrir con xarray
    ds = None
    for engine in ["netcdf4", "h5netcdf", "scipy"]:
        try:
            ds = xr.open_dataset(real_path, engine=engine)
            break
        except Exception:
            continue
    if tmp_dir:
        shutil.rmtree(tmp_dir, ignore_errors=True)
    if ds is None:
        print(f"  ✗ No se pudo abrir {nc_path.name} — saltando")
        continue
    all_vars = list(ds.data_vars)
    print(f"  {nc_path.name}: vars={all_vars}")
    var = next((v for v in ["tcc", "total_cloud_cover"] if v in ds), None)
    if var is None:
        print(f"  ✗ Variable TCC no encontrada, vars={all_vars}")
        ds.close()
        continue
    df_tcc = (
        ds[var]
        .to_dataframe()
        .reset_index()
        .rename(columns={var: "tcc", "valid_time": "timestamp", "time": "timestamp"})
    )
    # Queda solo timestamp y tcc
    df_tcc = df_tcc[["timestamp", "tcc"]].copy()
    df_tcc["timestamp"] = pd.to_datetime(df_tcc["timestamp"]).dt.tz_localize(None)
    df_tcc["item_id"] = item_id
    tcc_list.append(df_tcc)
    ds.close()

df_tcc_all = pd.concat(tcc_list, ignore_index=True)
print(f"TCC cargado: {len(df_tcc_all):,} filas | plantas: {df_tcc_all['item_id'].nunique()}")

# ── 2. Cargar predicciones walk-forward ──────────────────────────────────────
archivos = sorted(PRED_DIR.glob("*.parquet"))
df_pred = pd.concat([pd.read_parquet(f) for f in archivos], ignore_index=True)
df_pred["timestamp"] = pd.to_datetime(df_pred["timestamp"]).dt.tz_localize(None)
print(f"Predicciones cargadas: {len(df_pred):,} filas")

# ── 3. Join pred ↔ TCC por (item_id, timestamp) ─────────────────────────────
df = df_pred.merge(df_tcc_all, on=["item_id", "timestamp"], how="left")
n_missing = df["tcc"].isna().sum()
print(f"Filas sin TCC: {n_missing:,} ({100*n_missing/len(df):.1f}%)")
df = df.dropna(subset=["tcc", "potencia_mw", "0.5"])

# ── 4. Clasificar régimen de nubosidad ───────────────────────────────────────
df["regimen"] = pd.cut(df["tcc"], bins=BINS, labels=LABELS)
print("\nDistribución de regímenes:")
print(df.groupby("regimen", observed=True).size())

# ── 5. nMAE por modelo × régimen ────────────────────────────────────────────
CAPACIDADES = df_static.set_index("item_id")["pot_inst_mw"].to_dict()
df["cap"]     = df["item_id"].map(CAPACIDADES)
df["ae_norm"] = (df["potencia_mw"] - df["0.5"]).abs() / df["cap"]

resultados = []
for modelo, gm in df.groupby("model"):
    for reg, gr in gm.groupby("regimen", observed=True):
        nmae = gr["ae_norm"].mean()
        resultados.append({"model": modelo, "regimen": str(reg), "nMAE": nmae})

df_res = pd.DataFrame(resultados)
pivot  = df_res.pivot(index="model", columns="regimen", values="nMAE")[LABELS]
pivot  = pivot.reindex([m for m in MODELOS_ORDEN if m in pivot.index])

print("\n=== nMAE por modelo × régimen ===")
print((pivot * 100).round(2).to_string())

# ── 6. Heatmap ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5.5), facecolor="white")
sns.heatmap(
    pivot * 100,
    cmap="YlOrRd", annot=True, fmt=".2f",
    annot_kws={"size": 9},
    linewidths=0.4, linecolor="white",
    cbar_kws={"label": "nMAE (%)"},
    ax=ax
)
ax.set_title(
    "nMAE (%) por modelo y régimen de nubosidad\n"
    "11 plantas SEIN-Perú · Walk-forward",
    fontsize=11, pad=10
)
ax.set_xlabel("Régimen de nubosidad (TCC ERA5)", fontsize=10)
ax.set_ylabel("")
plt.setp(ax.get_xticklabels(), rotation=0, fontsize=10)
plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "heatmap_nmae_nubosidad.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT_DIR / "heatmap_nmae_nubosidad.pdf", bbox_inches="tight")
plt.show()

# ── 7. Barras — los 3 mejores modelos ────────────────────────────────────────
TOP3 = pivot.mean(axis=1).nsmallest(3).index.tolist()
df_top3 = df_res[df_res["model"].isin(TOP3)].copy()
df_top3["regimen"] = pd.Categorical(df_top3["regimen"], categories=LABELS, ordered=True)
df_top3 = df_top3.sort_values("regimen")

COLORES_BAR = {
    "WeightedEnsemble": "#1a6faf",
    "DirectTabular":    "#e05c00",
    "Chronos2":         "#2ca02c",
    "Chronos2SmallFineTuned":           "#9467bd",
    "ChronosWithRegressor[bolt_small]": "#8c564b",
}

fig, ax = plt.subplots(figsize=(8, 5), facecolor="white")
x      = np.arange(len(LABELS))
n_top  = len(TOP3)
width  = 0.22
offset = np.linspace(-(n_top-1)/2, (n_top-1)/2, n_top) * width

for i, modelo in enumerate(TOP3):
    dm = df_top3[df_top3["model"] == modelo].set_index("regimen")["nMAE"].reindex(LABELS) * 100
    col = COLORES_BAR.get(modelo, f"C{i}")
    ax.bar(x + offset[i], dm.values, width,
           label=modelo, color=col, edgecolor="black", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(LABELS, fontsize=11)
ax.set_ylabel("nMAE (%)", fontsize=11)
ax.set_title(
    "nMAE por régimen de nubosidad — 3 mejores modelos\n"
    "11 plantas SEIN-Perú · Walk-forward",
    fontsize=11, pad=10
)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "barras_nmae_nubosidad_top3.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT_DIR / "barras_nmae_nubosidad_top3.pdf", bbox_inches="tight")
plt.show()

print(f"\n✓ Guardado en {OUT_DIR}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

OUT_DIR = Path("models/walkforward_v1/analysis")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def tabla_a_png(df, titulo, nombre_archivo, fmt=".4f"):
    """Exporta un DataFrame como imagen PNG lista para adjuntar."""
    fig, ax = plt.subplots(figsize=(max(8, df.shape[1]*1.6), max(2.5, df.shape[0]*0.45 + 1.2)),
                           facecolor="white")
    ax.axis("off")
    tabla = ax.table(
        cellText=df.applymap(lambda x: f"{x:{fmt}}" if isinstance(x, float) else str(x)).values,
        colLabels=df.columns.tolist(),
        rowLabels=df.index.tolist(),
        cellLoc="center", loc="center"
    )
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(9)
    tabla.scale(1, 1.6)
    # Header en azul oscuro
    for j in range(len(df.columns)):
        tabla[0, j].set_facecolor("#0F2C4C")
        tabla[0, j].set_text_props(color="white", fontweight="bold")
    # Filas alternas
    for i in range(1, len(df) + 1):
        color = "#F4F8FB" if i % 2 == 0 else "white"
        for j in range(-1, len(df.columns)):
            tabla[i, j].set_facecolor(color)
    ax.set_title(titulo, fontsize=11, pad=12, fontweight="bold")
    plt.tight_layout()
    path = OUT_DIR / nombre_archivo
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"✓ {path}")

# ── 1. Leaderboard walk-forward (media entre plantas) ────────────────────────
# Asume que tienes df_metrics_mean con las métricas del walk-forward
tabla_a_png(
    resumen_mean[["nMAE","nRMSE","WQL","SQL","R"]].round(4),
    "Leaderboard — Walk-forward · Media entre plantas · 11 plantas SEIN-Perú",
    "tabla_leaderboard_media.png"
)

# ── 2. Leaderboard mediana ────────────────────────────────────────────────────
tabla_a_png(
    resumen[["nMAE","nRMSE","WQL","SQL","R"]].round(4),
    "Leaderboard — Walk-forward · Mediana entre plantas · 11 plantas SEIN-Perú",
    "tabla_leaderboard_mediana.png"
)

# ── 3. Permutation Importance por modelo ─────────────────────────────────────
df_pi = pd.read_csv(OUT_DIR / "perm_importance_por_modelo.csv", index_col=0)
tabla_a_png(
    df_pi.round(4),
    "Permutation Importance — Covariables meteorológicas · Walk-forward",
    "tabla_perm_importance.png"
)

# ── 4. nMAE por régimen de nubosidad ─────────────────────────────────────────
df_nub = pd.read_csv(OUT_DIR / "perm_importance_tabla.csv", index_col=0) \
           if (OUT_DIR / "perm_importance_tabla.csv").exists() else None
# La tabla de nubosidad está en el pivot — recalcular desde el CSV si existe
# (si no, corre la celda de nubosidad primero)

In [ ]:
%pip install openmeteo-requests requests-cache retry-requests -q

In [ ]:
# descargar_openmeteo.py
import openmeteo_requests, requests_cache, pandas as pd, numpy as np
from retry_requests import retry
from openmeteo_sdk.Variable import Variable
from pathlib import Path

OUT = Path("data/raw/openmeteo_hf"); OUT.mkdir(parents=True, exist_ok=True)
cache = requests_cache.CachedSession(".om_cache", expire_after=-1)
om = openmeteo_requests.Client(session=retry(cache, retries=5, backoff_factor=0.3))

URL = "https://historical-forecast-api.open-meteo.com/v1/forecast"
# ORDEN IMPORTA: así se leen abajo por índice
HOURLY = ["shortwave_radiation", "temperature_2m", "relative_humidity_2m",
          "wind_speed_10m", "surface_pressure", "precipitation"]

def descargar(item_id, lat, lon, start="2022-01-01", end="2026-05-31"):
    out = OUT / f"{item_id}.parquet"
    if out.exists():
        print(f"  {item_id:14} existe, salto"); return
    params = {"latitude": lat, "longitude": lon,
              "start_date": start, "end_date": end,
              "hourly": HOURLY, "timezone": "America/Lima"}
    resp = om.weather_api(URL, params=params)[0]
    h = resp.Hourly()
    # eje temporal
    idx = pd.date_range(
        start=pd.to_datetime(h.Time(),   unit="s", utc=True),
        end  =pd.to_datetime(h.TimeEnd(),unit="s", utc=True),
        freq =pd.Timedelta(seconds=h.Interval()), inclusive="left")
    # variables EN EL MISMO ORDEN que HOURLY
    data = {"timestamp": idx}
    for i, name in enumerate(["ghi","t_air","hr","wind_speed","pressure","precip_mm"]):
        data[name] = h.Variables(i).ValuesAsNumpy()
    df = pd.DataFrame(data)
    df["timestamp"] = df["timestamp"].dt.tz_convert("America/Lima").dt.tz_localize(None)
    df["wind_speed"] = df["wind_speed"] / 3.6      # km/h -> m/s (ERA5 en m/s)
    df.to_parquet(out, index=False)
    print(f"  {item_id:14} OK  {len(df)} filas")

# df_static ya tiene item_id, latitude, longitude
for _, r in df_static.iterrows():
    descargar(r["item_id"], r["latitude"], r["longitude"])

In [ ]:
import pandas as pd
from pathlib import Path

om = pd.read_parquet("data/raw/openmeteo_hf/TACNA.parquet")
print(om.describe())
print("\nRango temporal:", om.timestamp.min(), "→", om.timestamp.max())
print("NaN por columna:\n", om.isna().sum())

In [ ]:
# === Bias correction ERA5 -> Open-Meteo (MBCn) — idempotente ===
import src.sources.mbcn_biascorr_v2 as mbcn
import pandas as pd, numpy as np, pickle
from pathlib import Path

BC_DIR = Path("models/biascorr_v1"); BC_DIR.mkdir(parents=True, exist_ok=True)
DIAG_PKL = BC_DIR / "diagnosticos.pkl"
OM_DIR   = Path("data/raw/openmeteo_hf")
COVARS_BC = ["ghi","t_air","hr","wind_speed","pressure","precip_mm"]

if DIAG_PKL.exists():
    with open(DIAG_PKL, "rb") as f:
        diagnosticos = pickle.load(f)
    print("Bias correction cacheado — cargando.\n")
else:
    diagnosticos = {}
    # ERA5 desde largo_unificado (ya en memoria)
    era5_all = largo_unificado.copy()
    era5_all["timestamp"] = pd.to_datetime(era5_all["timestamp"])

    for planta in era5_all.item_id.unique():
        # --- ERA5 de la planta ---
        e = (era5_all[era5_all.item_id == planta]
             .set_index("timestamp")[COVARS_BC].copy())
        e["hour"] = e.index.hour
        # --- Open-Meteo de la planta ---
        o = pd.read_parquet(OM_DIR / f"{planta}.parquet").set_index("timestamp")
        o["hour"] = o.index.hour
        # --- SOLAPE: timestamps comunes ---
        idx = e.index.intersection(o.index)
        if len(idx) < 2000:
            print(f"  {planta:14} solape insuficiente ({len(idx)}h), salto"); continue
        e_s = e.loc[idx]; o_s = o.loc[idx, COVARS_BC + ["hour"]]
        # --- descartar filas con NaN en cualquiera ---
        ok = e_s[COVARS_BC].notna().all(1) & o_s[COVARS_BC].notna().all(1)
        e_s, o_s = e_s[ok], o_s[ok]

        om_corr, diag = mbcn.corregir_planta(e_s, o_s)
        diagnosticos[planta] = diag
        om_corr.to_parquet(BC_DIR / f"{planta}_corr.parquet")
        print(f"  {planta:14} corregida  ({ok.sum()}h)")

    with open(DIAG_PKL, "wb") as f:
        pickle.dump(diagnosticos, f)
    print()

# === Tabla KS (mediana entre plantas) — continuas ===
print("KS mediana entre plantas (continuas):")
print(f"{'covariable':<12}{'KS crudo':>12}{'KS corregido':>16}")
print("-"*40)
for c in [v for v in COVARS_BC if v != "precip_mm"]:
    raw = np.median([diagnosticos[p][c]["ks_raw"] for p in diagnosticos])
    cor = np.median([diagnosticos[p][c]["ks_cor"] for p in diagnosticos])
    print(f"{c:<12}{raw:>12.4f}{cor:>16.4f}")

# === Precipitación (dos componentes) ===
efs_raw = np.median([diagnosticos[p]["precip_mm"]["err_frac_seca_raw"] for p in diagnosticos])
efs_cor = np.median([diagnosticos[p]["precip_mm"]["err_frac_seca_cor"] for p in diagnosticos])
ksi_raw = np.nanmedian([diagnosticos[p]["precip_mm"]["ks_intensidad_raw"] for p in diagnosticos])
ksi_cor = np.nanmedian([diagnosticos[p]["precip_mm"]["ks_intensidad_cor"] for p in diagnosticos])
print("\nPrecipitación (mediana entre plantas):")
print(f"  Error frecuencia seca:   {efs_raw:.4f}  ->  {efs_cor:.4f}")
print(f"  KS intensidad (húmedos): {ksi_raw:.4f}  ->  {ksi_cor:.4f}")

In [ ]:
# KS del GHI solo diurno (ghi>0 en ambas fuentes), mediana entre plantas
from scipy.stats import ks_2samp
ks_ghi_raw, ks_ghi_cor = [], []
for planta in diagnosticos:
    e = largo_unificado[largo_unificado.item_id==planta].set_index("timestamp")
    o = pd.read_parquet(f"data/raw/openmeteo_hf/{planta}.parquet").set_index("timestamp")
    oc = pd.read_parquet(f"models/biascorr_v1/{planta}_corr.parquet")
    idx = e.index.intersection(o.index)
    d = (e.loc[idx,"ghi"]>0) & (o.loc[idx,"ghi"]>0)
    ks_ghi_raw.append(ks_2samp(o.loc[idx,"ghi"][d.values], e.loc[idx,"ghi"][d.values]).statistic)
    ks_ghi_cor.append(ks_2samp(oc["ghi"].values[:len(d)][d.values[:len(oc)]], e.loc[idx,"ghi"][d.values][:len(oc)]).statistic)
print(f"GHI diurno — KS crudo: {np.median(ks_ghi_raw):.4f}  corregido: {np.median(ks_ghi_cor):.4f}")

In [ ]:
import pickle
from pathlib import Path
p = Path("models/tres_condiciones_v1/nmae_3cond.pkl")
obj = pickle.load(open(p,"rb"))
print("tipo:", type(obj))
print(obj if not hasattr(obj,'head') else obj.head())
print("filas:", len(obj))

In [ ]:
# === Tabla 3 condiciones — usa la MECÁNICA de la celda 42 (probada) ===
import pandas as pd, numpy as np, pickle
from pathlib import Path
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

T3_PKL = Path("models/tres_condiciones_v1/nmae_3cond.pkl")
T3_PKL.parent.mkdir(parents=True, exist_ok=True)
COVARS = ["ghi","t_air","hr","wind_speed","pressure","precip_mm"]
CAPACIDADES_MW = df_static.set_index("item_id")["pot_inst_mw"].to_dict()

if T3_PKL.exists():
    df_res3 = pickle.load(open(T3_PKL, "rb"))
    if hasattr(df_res3, "groupby") and len(df_res3):
        print("Tabla 3 condiciones cacheada — sin recalcular.\n")
        print("nMAE mediana entre plantas:")
        print(df_res3.groupby("fuente")["nMAE"].median().round(4))
    else:
        print("Cache inválido (dict/vacío) — hay que recalcular: borrá el .pkl y corré la celda de cómputo.")
else:
    print("No hay cache — hay que correr la celda de cómputo completa.")

OM_DIR, BC_DIR = Path("data/raw/openmeteo_hf"), Path("models/biascorr_v1")
fits = sorted(Path("models/walkforward_v1/fits").glob("fit_*"))
predictor = TimeSeriesPredictor.load(str(fits[-1]))
M, PL = "WeightedEnsemble", 24

base = largo_unificado.sort_values(["item_id","timestamp"]).reset_index(drop=True).copy()
base["timestamp"] = pd.to_datetime(base["timestamp"])

def tsdf(df_):
    return TimeSeriesDataFrame.from_data_frame(
        df_, id_column="item_id", timestamp_column="timestamp", static_features_df=df_static)

# --- cargar covariables OM alineadas al MISMO índice/tz que base ---
# clave: reindexar contra los timestamps REALES de base, no un date_range genérico
def cov_por_planta(fuente):
    out = {}
    for p in base.item_id.unique():
        if fuente=="era5":
            g = base[base.item_id==p].set_index("timestamp")[COVARS]
        else:
            f = OM_DIR if fuente=="om_crudo" else BC_DIR
            fn = f/f"{p}.parquet" if fuente=="om_crudo" else f/f"{p}_corr.parquet"
            g = pd.read_parquet(fn)
            g = g.set_index("timestamp") if "timestamp" in g.columns else g
            g.index = pd.to_datetime(g.index)
        out[p] = g[COVARS]
    return out

# solape: primer día de OM + margen, hasta el final de base, muestreo semanal
om0 = pd.read_parquet(OM_DIR/f"{base.item_id.iloc[0]}.parquet")
t0 = pd.to_datetime(om0["timestamp"]).min().normalize() + pd.Timedelta(days=1)
t1 = base["timestamp"].max().normalize() - pd.Timedelta(days=1)
dias = pd.date_range(t0, t1, freq="D")[::7]
print(f"Días evaluados: {len(dias)} (solape, semanal)")

filas = []
for fuente in ["era5","om_crudo","om_corr"]:
    covs = cov_por_planta(fuente)
    preds = []
    for dia in dias:
        d0, d1 = dia, dia + pd.Timedelta(days=1)
        # FUTURO igual que celda 42: filtrar base por rango real
        fut = base[(base.timestamp >= d0) & (base.timestamp < d1)].copy()
        ctx = base[base.timestamp < d0]
        if fut.empty or ctx.empty: continue
        # sustituir covariables futuras por las de la fuente (join por planta+timestamp)
        fut = fut[["item_id","timestamp","potencia_mw"]].copy()
        cov_rows = []
        for p in fut.item_id.unique():
            sub = covs[p].reindex(fut[fut.item_id==p]["timestamp"])
            sub["item_id"] = p
            cov_rows.append(sub.reset_index().rename(columns={"index":"timestamp"}))
        cov_fut = pd.concat(cov_rows, ignore_index=True)
        fut_full = fut.merge(cov_fut, on=["item_id","timestamp"], how="inner")
        fut_full = fut_full.dropna(subset=COVARS)
        if fut_full.empty: continue
        ok = fut_full.item_id.unique()
        ctx_ok = ctx[ctx.item_id.isin(ok)]
        try:
            pm = predictor.predict(tsdf(ctx_ok),
                    known_covariates=tsdf(fut_full[["item_id","timestamp"]+COVARS]),
                    model=M).reset_index()
        except Exception as e:
            print(f"  {fuente} {dia.date()}: {repr(e)[:80]}"); continue
        pm = pm.merge(fut_full[["item_id","timestamp","potencia_mw"]],
                      on=["item_id","timestamp"], how="left")
        preds.append(pm)
    if not preds:
        print(f"  ⚠ {fuente}: 0 predicciones"); continue
    P = pd.concat(preds, ignore_index=True)
    for planta, gp in P.groupby("item_id"):
        cap = CAPACIDADES_MW.get(planta)
        if cap is None: continue
        yt, ym = gp["potencia_mw"].values, gp["0.5"].values
        m = ~np.isnan(yt) & ~np.isnan(ym)
        if m.sum()==0: continue
        filas.append({"fuente":fuente,"item_id":planta,
                      "nMAE":np.mean(np.abs(yt[m]-ym[m]))/cap})
    print(f"  {fuente}: {len(P)} predicciones, {P.item_id.nunique()} plantas")

df_res3 = pd.DataFrame(filas)
pickle.dump(df_res3, open(T3_PKL,"wb"))
print("\nnMAE mediana entre plantas:")
print(df_res3.groupby("fuente")["nMAE"].median().round(4) if len(df_res3) else "SIN DATOS")

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

# 1. Extraer un día real de Clemesi del pronóstico del WeightedEnsemble
PLANTA = "CLEMESI"
sub = df_pred[(df_pred["model"] == "WeightedEnsemble") &
              (df_pred["item_id"] == PLANTA)].copy()
sub["timestamp"] = pd.to_datetime(sub["timestamp"])
sub = sub.sort_values("timestamp")
sub["fecha"] = sub["timestamp"].dt.date
SOC_MIN = 30

# elegir día despejado: mayor generación observada acumulada (banda angosta)
DIA = sub.groupby("fecha")["potencia_mw"].sum().idxmax()   # cambiar a mano si querés otro
d = sub[sub["fecha"] == DIA].copy()
d = d.set_index(d["timestamp"].dt.hour).sort_index().reindex(range(24))

h   = d.index.values
q10 = d["0.1"].values
q50 = d["0.5"].values
q90 = d["0.9"].values
obs = d["potencia_mw"].values

# después de extraer q10/q50/q90/obs de Clemesi, normalizar y reescalar a residencial
PICO_RESIDENCIAL = 3.5   # kW, sistema PV residencial típico
factor = PICO_RESIDENCIAL / np.nanmax(q50)
q10, q50, q90, obs = q10*factor, q50*factor, q90*factor, obs*factor

# ahora la carga residencial real
P_CARGA = 2.0   # kW, termotanque
DELTA   = 0.2   # kW
HUL, HLL = P_CARGA + DELTA, P_CARGA - DELTA   # 2.2 / 1.8 kW
BAT_CAP = 5.0   # kWh

# 3. Simular SoC + decisión hora a hora sobre los cuantiles REALES
soc = np.full(24, np.nan); soc[0] = 55.0
estado = np.zeros(24, dtype=int); prev = 0
for t in range(24):
    if t > 0 and not np.isnan(q50[t]):
        consumo = P_CARGA if prev else 0.0
        net = (q50[t] - consumo) * 0.5
        base = soc[t-1] if not np.isnan(soc[t-1]) else 55.0
        soc[t] = np.clip(base + net / BAT_CAP * 100, 5, 100)
    elif t > 0:
        soc[t] = soc[t-1]
    qp = q10[t] if not np.isnan(q10[t]) else 0.0
    if qp >= HUL:            e = 1
    elif qp < HLL:           e = 1 if soc[t] > SOC_MIN else 0
    else:                    e = prev
    estado[t] = e; prev = e

# 4. Figura fondo blanco
plt.style.use("default")
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.2, 5.2), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1.1]})
fig.patch.set_facecolor("white")

ax1.fill_between(h, q10, q90, alpha=0.18, color="#1f6feb", label=r"Banda $[q_{0.1},\,q_{0.9}]$")
ax1.plot(h, q50, color="#1f6feb", lw=1.8, label=r"Mediana $q_{0.5}$")
ax1.plot(h, q10, color="#0a3d91", lw=1.3, ls="--", label=r"$q_{0.1}$ (piso)")
ax1.plot(h, obs, color="#111", lw=1.0, ls=":", label="Generación observada")
ax1.axhline(HUL, color="#c0392b", lw=1, ls="-."); ax1.axhline(HLL, color="#e67e22", lw=1, ls="-.")
ax1.text(23.2, HUL, "HUL", color="#c0392b", va="center", fontsize=8)
ax1.text(23.2, HLL, "HLL", color="#e67e22", va="center", fontsize=8)
for t in range(24):
    if estado[t]: ax1.axvspan(t-0.5, t+0.5, color="#2ecc71", alpha=0.10, lw=0)
ax1.set_ylabel("Potencia [kW]")
ax1.set_title(f"Pronóstico probabilístico gobernando la carga diferible — {PLANTA.title()} ({DIA})", fontsize=9.5)
ax1.legend(fontsize=7.5, loc="upper left", ncol=2, framealpha=0.9)
ax1.set_ylim(0, np.nanmax(q90) * 1.15)

ax2b = ax2.twinx()
ax2.step(h, estado, where="mid", color="#27ae60", lw=1.8)
ax2.fill_between(h, 0, estado, step="mid", color="#2ecc71", alpha=0.15)
ax2b.plot(h, soc, color="#8e44ad", lw=1.4)
ax2b.axhline(SOC_MIN, color="#8e44ad", lw=0.8, ls=":")
ax2b.text(0.2, SOC_MIN+3, r"$SoC_{min}$", color="#8e44ad", fontsize=8)
ax2.set_yticks([0, 1]); ax2.set_yticklabels(["OFF", "ON"])
ax2.set_ylabel("Relé"); ax2b.set_ylabel("SoC [%]", color="#8e44ad")
ax2b.set_ylim(0, 100); ax2.set_ylim(-0.15, 1.35)
ax2.set_xlabel("Hora del día (UTC-5)"); ax2.set_xticks(range(0, 24, 3))

fig.tight_layout()
fig.savefig("reports/lazo_cerrado_clemesi.png", dpi=200, bbox_inches="tight", facecolor="white")
print(f"Día usado: {DIA} | pico q0.5={np.nanmax(q50):.3f} MW | P_carga={P_CARGA:.3f} MW | horas ON={int(np.nansum(estado))}")

In [ ]:
# === EXPERIMENTO AISLADO: predicción sobre planta excluida, sin reentrenar ===
PLANTA_EXTRA = "SAN_MARTIN"  # <-- elegí una de las EXCLUIDAS (ver celda 14)

_archivos_e = sorted(glob.glob(f"{RAW_DIR}/*.csv"))
_partes_e = [leer_csv_mes(a) for a in _archivos_e]
_full_e = pd.concat(_partes_e).sort_index()
_full_e = _full_e[~_full_e.index.duplicated(keep="first")]
_ancho_e = _full_e.resample("1h", label="right", closed="right").mean()

largo_extra = (_ancho_e.reset_index()
               .melt(id_vars="timestamp", var_name="item_id", value_name="potencia_mw")
               .dropna(subset=["potencia_mw"]))
largo_extra = largo_extra[largo_extra.item_id == PLANTA_EXTRA].sort_values("timestamp").reset_index(drop=True)
del _partes_e, _full_e, _ancho_e

if largo_extra.empty:
    raise SystemExit(f"'{PLANTA_EXTRA}' no aparece en MAPEO (celda 12) o en los CSVs. Revisar nombre.")
print(f"{PLANTA_EXTRA}: {len(largo_extra)} filas, {largo_extra.timestamp.min()} -> {largo_extra.timestamp.max()}")

In [ ]:
r_e = requests.get(URL_GIS, params=params, timeout=60)
gis_e = {feat["attributes"]["NOMBRE"]: feat["attributes"] for feat in r_e.json()["features"]}

print("=== Todas las centrales solares en el GIS ===")
for nombre in sorted(gis_e.keys()):
    print(f"  {nombre!r}")

# Búsqueda insensible a mayúsculas/tildes por substring, no exacta
import unicodedata
def _sin_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFD", s)
                   if unicodedata.category(c) != "Mn").upper()

palabra_clave = "SAN MARTIN"  # <-- ajustá esto si buscás otra planta
candidatos = [n for n in gis_e if palabra_clave in _sin_tildes(n)]
print(f"\nCandidatos para '{palabra_clave}': {candidatos}")

if len(candidatos) != 1:
    raise SystemExit(
        f"Encontré {len(candidatos)} candidatos, no 1 -- "
        f"elegí a mano de la lista de arriba y asigná nombre_gis_e manualmente."
    )

nombre_gis_e = candidatos[0]
info_e = gis_e[nombre_gis_e]
lat_e, lon_e, pot_inst_e = info_e["LATITUD"], info_e["LONGITUD"], info_e["POT_INST"]
inicio_e, fin_e = largo_extra["timestamp"].min(), largo_extra["timestamp"].max()
print(f"\nUsando: {nombre_gis_e!r} -> lat={lat_e:.4f} lon={lon_e:.4f} pot_inst={pot_inst_e} MW")

In [ ]:
print(f"Máximo observado en COES: {largo_extra['potencia_mw'].max():.1f} MW")
print(f"POT_INST del GIS:         {pot_inst_e:.1f} MW")

In [ ]:
descargar_planta_timeseries(client, item_id=PLANTA_EXTRA, lat=lat_e, lon=lon_e, inicio=inicio_e, fin=fin_e)
descargar_tcc(client, item_id=PLANTA_EXTRA, lat=lat_e, lon=lon_e, inicio=inicio_e, fin=fin_e)

In [ ]:
df_meteo_e = leer_era5_planta(PLANTA_EXTRA)
if df_meteo_e is None:
    raise SystemExit("Falló la descarga ERA5-Land de la celda anterior.")

df_meteo_e["ghi"]        = df_meteo_e["ssrd"] / 3600.0
df_meteo_e["t_air"]      = df_meteo_e["t2m"] - 273.15
df_meteo_e["dewpoint"]   = df_meteo_e["d2m"] - 273.15
a, b = 17.625, 243.04
Td, T = df_meteo_e["dewpoint"], df_meteo_e["t_air"]
df_meteo_e["hr"]         = 100.0 * np.exp((a*Td)/(b+Td)) / np.exp((a*T)/(b+T))
df_meteo_e["wind_speed"] = np.sqrt(df_meteo_e["u10"]**2 + df_meteo_e["v10"]**2)
df_meteo_e["pressure"]   = df_meteo_e["sp"] / 100.0
df_meteo_e["precip_mm"]  = df_meteo_e["tp"] * 1000.0
df_meteo_e["timestamp"]  = df_meteo_e["timestamp"] - pd.Timedelta(hours=5)
df_meteo_e = df_meteo_e[["timestamp","item_id","ghi","t_air","hr","wind_speed","pressure","precip_mm"]]

largo_unificado_extra = pd.merge(
    largo_extra, df_meteo_e, on=["timestamp","item_id"], how="inner"
).sort_values("timestamp").reset_index(drop=True)
print(f"largo_unificado_extra: {largo_unificado_extra.shape}")

In [ ]:
alt_e = obtener_elevacion(lat_e, lon_e)
df_static_extra = pd.DataFrame([{
    "item_id": PLANTA_EXTRA, "latitude": lat_e, "longitude": lon_e,
    "pot_inst_mw": pot_inst_e, "altitude_m": alt_e,
}])

faltantes = [c for c in df_static.columns if c not in df_static_extra.columns]
for col in faltantes:
    df_static_extra[col] = np.nan

df_static_extra = df_static_extra[df_static.columns]  # mismo orden, "item_id" ya es columna
print(df_static_extra)

In [ ]:
tsdf_extra = TimeSeriesDataFrame.from_data_frame(
    largo_unificado_extra, id_column="item_id", timestamp_column="timestamp",
    static_features_df=df_static_extra,
)
FIT_DIR = Path("models/walkforward_v1/fits")
ultimo_fit = sorted(FIT_DIR.glob("fit_*"))[-1]
print(f"Cargando (sin reentrenar): {ultimo_fit}")
predictor_cargado = TimeSeriesPredictor.load(str(ultimo_fit))

In [ ]:
print(sorted(FIT_DIR.glob("fit_*")))

In [ ]:
COVARS = ["ghi","t_air","hr","wind_speed","pressure","precip_mm"]
N_DIAS_TEST = 7  # ventana de prueba, igual espíritu que las 168h del resto del paper

def tsdf_e(df_):
    return TimeSeriesDataFrame.from_data_frame(
        df_, id_column="item_id", timestamp_column="timestamp",
        static_features_df=df_static_extra,
    )

completo = largo_unificado_extra.sort_values("timestamp").reset_index(drop=True)
t_max_e = completo["timestamp"].max().normalize()
dias_test = pd.date_range(t_max_e - pd.Timedelta(days=N_DIAS_TEST), t_max_e - pd.Timedelta(days=1), freq="D")

modelos_e = predictor_cargado.model_names()
print(f"Modelos en el predictor cargado: {modelos_e}")

preds_e = []
for dia in dias_test:
    ctx_df_e = completo[completo["timestamp"] < dia]
    fut_df_e = completo[(completo["timestamp"] >= dia) & (completo["timestamp"] < dia + pd.Timedelta(days=1))]
    if fut_df_e.empty or ctx_df_e.empty:
        continue
    ctx_ts_e   = tsdf_e(ctx_df_e)
    fut_covs_e = tsdf_e(fut_df_e[["item_id","timestamp"] + COVARS])
    act_e      = fut_df_e[["item_id","timestamp","potencia_mw"]]

    for m in modelos_e:
        try:
            pm = predictor_cargado.predict(ctx_ts_e, known_covariates=fut_covs_e, model=m).reset_index()
            pm["model"] = m
            pm = pm.merge(act_e, on=["item_id","timestamp"], how="left")
            preds_e.append(pm)
        except Exception as e:
            print(f"  {dia.date()} {m}: FALLÓ -- {type(e).__name__}: {e}")

if not preds_e:
    raise SystemExit("Ningún modelo predijo sobre la planta nueva sin reentrenar -- hallazgo válido igual.")

df_pred_extra = pd.concat(preds_e, ignore_index=True)
OUT_EXTRA = Path("models/coldstart_extra")
OUT_EXTRA.mkdir(parents=True, exist_ok=True)
df_pred_extra.to_parquet(OUT_EXTRA / f"{PLANTA_EXTRA}_pred.parquet")
print(f"\n{len(df_pred_extra)} filas, modelos: {sorted(df_pred_extra['model'].unique())}")

In [ ]:
QUANTILES_E = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
resultados_e = []
for modelo, gp in df_pred_extra.sort_values("timestamp").groupby("model"):
    yt, ym, ymean = gp["potencia_mw"].values, gp["0.5"].values, gp["mean"].values
    mask = ~np.isnan(yt) & ~np.isnan(ym)
    if mask.sum() == 0: continue
    yt_, ym_, ymean_ = yt[mask], ym[mask], ymean[mask]
    mae, rmse = np.mean(np.abs(yt_-ym_)), np.sqrt(np.mean((ymean_-yt_)**2))
    pinballs = [np.mean(np.where(yt_>=gp[str(q)].values[mask], q*(yt_-gp[str(q)].values[mask]),
                (1-q)*(gp[str(q)].values[mask]-yt_))) for q in QUANTILES_E]
    wql = 2*np.mean(pinballs)/(np.mean(yt_)+1e-8)
    y_naive = gp["potencia_mw"].shift(24).values[mask]
    mask2 = ~np.isnan(y_naive)
    sql = 2*np.mean(pinballs)/(np.mean(np.abs(yt_[mask2]-y_naive[mask2]))+1e-8) if mask2.sum()>0 else np.nan
    resultados_e.append({"model":modelo,"nMAE":mae/pot_inst_e,"nRMSE":rmse/pot_inst_e,"WQL":wql,"SQL":sql,"n":mask.sum()})

df_res_e = pd.DataFrame(resultados_e).sort_values("nMAE")
print(f"\n=== Cold-start ({PLANTA_EXTRA}), {N_DIAS_TEST} días, inferencia sin reentrenar ===")
print(df_res_e.round(4).to_string(index=False))
print("\nComparar contra Tabla 2 (11 plantas): WeightedEnsemble nMAE=0.0398 SQL=0.6841")

In [ ]:
# === Cold-start generalizado: reusa todo lo ya resuelto para SAN_MARTIN ===
import unicodedata

def _sin_tildes(s):
    return "".join(c for c in unicodedata.normalize("NFD", s)
                   if unicodedata.category(c) != "Mn").upper()

COVARS = ["ghi","t_air","hr","wind_speed","pressure","precip_mm"]
N_DIAS_TEST = 4
MODELOS_INTERES = ["WeightedEnsemble", "DirectTabular", "Chronos2"]
QUANTILES_E = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]

def preparar_planta(planta_extra, palabra_clave_gis):
    """Pasos 1-2: generación COES + coordenadas/potencia GIS. Devuelve None si no hay match único."""
    global _partes_e, _full_e, _ancho_e
    _archivos_e = sorted(glob.glob(f"{RAW_DIR}/*.csv"))
    _partes_e = [leer_csv_mes(a) for a in _archivos_e]
    _full_e = pd.concat(_partes_e).sort_index()
    _full_e = _full_e[~_full_e.index.duplicated(keep="first")]
    _ancho_e = _full_e.resample("1h", label="right", closed="right").mean()
    largo_e = (_ancho_e.reset_index().melt(id_vars="timestamp", var_name="item_id", value_name="potencia_mw")
               .dropna(subset=["potencia_mw"]))
    largo_e = largo_e[largo_e.item_id == planta_extra].sort_values("timestamp").reset_index(drop=True)
    del _partes_e, _full_e, _ancho_e
    if largo_e.empty:
        print(f"  [{planta_extra}] SIN generación en COES/MAPEO -- salto.")
        return None

    r_e = requests.get(URL_GIS, params=params, timeout=60)
    gis_e = {feat["attributes"]["NOMBRE"]: feat["attributes"] for feat in r_e.json()["features"]}
    candidatos = [n for n in gis_e if palabra_clave_gis in _sin_tildes(n)]
    if len(candidatos) != 1:
        print(f"  [{planta_extra}] {len(candidatos)} candidatos GIS para '{palabra_clave_gis}': {candidatos} -- salto, requiere resolución manual.")
        return None

    info_e = gis_e[candidatos[0]]
    return {
        "item_id": planta_extra, "largo": largo_e,
        "lat": info_e["LATITUD"], "lon": info_e["LONGITUD"], "pot_inst": info_e["POT_INST"],
        "gis_nombre": candidatos[0],
    }

def evaluar_planta(planta_extra, palabra_clave_gis):
    print(f"\n=== {planta_extra} ===")
    info = preparar_planta(planta_extra, palabra_clave_gis)
    if info is None:
        return None
    lat_e, lon_e, pot_inst_e = info["lat"], info["lon"], info["pot_inst"]
    largo_e = info["largo"]
    inicio_e, fin_e = largo_e["timestamp"].min(), largo_e["timestamp"].max()

    # chequeo de coherencia (el mismo que hicimos a mano para San Martín)
    max_obs = largo_e["potencia_mw"].max()
    print(f"  Máximo observado COES: {max_obs:.1f} MW | POT_INST GIS: {pot_inst_e:.1f} MW ({info['gis_nombre']})")
    if max_obs > pot_inst_e * 1.15:
        print(f"  AVISO: máximo observado supera la potencia instalada por >15% -- revisar antes de confiar en pot_inst_e.")

    descargar_planta_timeseries(client, item_id=planta_extra, lat=lat_e, lon=lon_e, inicio=inicio_e, fin=fin_e)
    descargar_tcc(client, item_id=planta_extra, lat=lat_e, lon=lon_e, inicio=inicio_e, fin=fin_e)

    df_meteo_e = leer_era5_planta(planta_extra)
    if df_meteo_e is None:
        print(f"  [{planta_extra}] Falló la descarga ERA5-Land -- salto."); return None
    df_meteo_e["ghi"]        = df_meteo_e["ssrd"] / 3600.0
    df_meteo_e["t_air"]      = df_meteo_e["t2m"] - 273.15
    df_meteo_e["dewpoint"]   = df_meteo_e["d2m"] - 273.15
    a, b = 17.625, 243.04
    Td, T = df_meteo_e["dewpoint"], df_meteo_e["t_air"]
    df_meteo_e["hr"]         = 100.0 * np.exp((a*Td)/(b+Td)) / np.exp((a*T)/(b+T))
    df_meteo_e["wind_speed"] = np.sqrt(df_meteo_e["u10"]**2 + df_meteo_e["v10"]**2)
    df_meteo_e["pressure"]   = df_meteo_e["sp"] / 100.0
    df_meteo_e["precip_mm"]  = df_meteo_e["tp"] * 1000.0
    df_meteo_e["timestamp"]  = df_meteo_e["timestamp"] - pd.Timedelta(hours=5)
    df_meteo_e = df_meteo_e[["timestamp","item_id","ghi","t_air","hr","wind_speed","pressure","precip_mm"]]
    largo_unificado_e = pd.merge(largo_e, df_meteo_e, on=["timestamp","item_id"], how="inner").sort_values("timestamp").reset_index(drop=True)

    alt_e = obtener_elevacion(lat_e, lon_e)
    df_static_e = pd.DataFrame([{"item_id": planta_extra, "latitude": lat_e, "longitude": lon_e,
                                  "pot_inst_mw": pot_inst_e, "altitude_m": alt_e}])
    for col in [c for c in df_static.columns if c not in df_static_e.columns]:
        df_static_e[col] = np.nan
    df_static_e = df_static_e[df_static.columns]

    def tsdf_e(df_):
        return TimeSeriesDataFrame.from_data_frame(df_, id_column="item_id", timestamp_column="timestamp",
                                                    static_features_df=df_static_e)

    completo = largo_unificado_e.sort_values("timestamp").reset_index(drop=True)
    t_max_e = completo["timestamp"].max().normalize()
    dias_test = pd.date_range(t_max_e - pd.Timedelta(days=N_DIAS_TEST), t_max_e - pd.Timedelta(days=1), freq="D")
    modelos_e = [m for m in MODELOS_INTERES if m in predictor_cargado.model_names()]

    preds_e = []
    for dia in dias_test:
        ctx_df_e = completo[completo["timestamp"] < dia]
        fut_df_e = completo[(completo["timestamp"] >= dia) & (completo["timestamp"] < dia + pd.Timedelta(days=1))]
        if fut_df_e.empty or ctx_df_e.empty: continue
        ctx_ts_e   = tsdf_e(ctx_df_e)
        fut_covs_e = tsdf_e(fut_df_e[["item_id","timestamp"] + COVARS])
        act_e      = fut_df_e[["item_id","timestamp","potencia_mw"]]
        for m in modelos_e:
            try:
                pm = predictor_cargado.predict(ctx_ts_e, known_covariates=fut_covs_e, model=m).reset_index()
                pm["model"] = m
                preds_e.append(pm.merge(act_e, on=["item_id","timestamp"], how="left"))
            except Exception as e:
                print(f"  {dia.date()} {m}: FALLÓ -- {type(e).__name__}: {e}")
    if not preds_e:
        print(f"  [{planta_extra}] Ningún modelo predijo -- salto."); return None

    df_pred_e = pd.concat(preds_e, ignore_index=True)
    resultados = []
    for modelo, gp in df_pred_e.sort_values("timestamp").groupby("model"):
        yt, ym, ymean = gp["potencia_mw"].values, gp["0.5"].values, gp["mean"].values
        mask = ~np.isnan(yt) & ~np.isnan(ym)
        if mask.sum() == 0: continue
        yt_, ym_, ymean_ = yt[mask], ym[mask], ymean[mask]
        mae, rmse = np.mean(np.abs(yt_-ym_)), np.sqrt(np.mean((ymean_-yt_)**2))
        pinballs = [np.mean(np.where(yt_>=gp[str(q)].values[mask], q*(yt_-gp[str(q)].values[mask]),
                    (1-q)*(gp[str(q)].values[mask]-yt_))) for q in QUANTILES_E]
        wql = 2*np.mean(pinballs)/(np.mean(yt_)+1e-8)
        y_naive = gp["potencia_mw"].shift(24).values[mask]
        mask2 = ~np.isnan(y_naive)
        sql = 2*np.mean(pinballs)/(np.mean(np.abs(yt_[mask2]-y_naive[mask2]))+1e-8) if mask2.sum()>0 else np.nan
        resultados.append({"planta": planta_extra, "model": modelo, "nMAE": mae/pot_inst_e,
                            "nRMSE": rmse/pot_inst_e, "WQL": wql, "SQL": sql, "n": mask.sum()})
    return pd.DataFrame(resultados)

# --- Correr sobre las 4 restantes ---
CANDIDATAS = [
    ("EXPANSION_INTIPAMPA", "INTIPAMPA"),
    ("COENERGY",            "COENERGY"),
    ("SUNNY",               "SUNNY"),
    ("SUNNY_EXPANSION",     "SUNNY"),
]
resultados_totales = []
for planta, palabra in CANDIDATAS:
    r = evaluar_planta(planta, palabra)
    if r is not None:
        resultados_totales.append(r)

if resultados_totales:
    df_final = pd.concat(resultados_totales, ignore_index=True).sort_values(["planta","nMAE"])
    print("\n" + df_final.round(4).to_string(index=False))
else:
    print("\nNinguna de las 4 se pudo evaluar automáticamente.")

In [ ]:
ID_ELEMENTO_COENERGY = 26899

s = make_session()
r, err = bajar_ficha(s, ID_ELEMENTO_COENERGY)
if err or not es_excel(r):
    print(f"Falló: {err or 'no es Excel'}")
else:
    ruta_coenergy = Path("data/raw/fichas/COENERGY.xlsx")
    ruta_coenergy.write_bytes(r.content)
    ficha_coenergy = parsear_ficha(ruta_coenergy)
    pot_inst_coenergy = ficha_coenergy["pot_inst_mw"]
    print(f"Potencia instalada COENERGY (de su propia ficha): {pot_inst_coenergy} MW")
    print(ficha_coenergy)  # imprimí el dict completo, puede traer ubicación

In [ ]:
# === Experimento C: TRI (Riley 1999) vs nMAE por planta ===
import time
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

NMAE_TABLA1 = {
    "CARHUAQUERO": 0.044, "CLEMESI": 0.035, "INTIPAMPA": 0.034, "MAJES": 0.022,
    "MATARANI": 0.040, "MOQUEGUA_FV": 0.043, "PANAMERICANA": 0.041,
    "REPARTICION": 0.023, "RUBI": 0.037, "TACNA": 0.056, "YARUCAYA": 0.041,
}
static_idx = df_static.set_index("item_id")
GRID_N, BOX_KM, KM_POR_GRADO = 5, 9.0, 111.0

def bbox_grid(lat, lon):
    half = BOX_KM / 2
    dlat = half / KM_POR_GRADO
    dlon = half / (KM_POR_GRADO * np.cos(np.radians(lat)))
    lats = np.linspace(lat-dlat, lat+dlat, GRID_N)
    lons = np.linspace(lon-dlon, lon+dlon, GRID_N)
    return [(la, lo) for la in lats for lo in lons]

def riley_tri(elev):
    n = elev.shape[0]; diffs = []
    for i in range(1, n-1):
        for j in range(1, n-1):
            vecinos = np.delete(elev[i-1:i+2, j-1:j+2].flatten(), 4)
            diffs.append(np.mean(np.abs(vecinos - elev[i,j])))
    return float(np.mean(diffs))

filas = []
for planta, nmae in NMAE_TABLA1.items():
    if planta not in static_idx.index:
        print(f"AVISO: {planta} no está en df_static"); continue
    lat, lon = static_idx.loc[planta, ["latitude","longitude"]]
    elevs = []
    for la, lo in bbox_grid(lat, lon):
        elevs.append(obtener_elevacion(la, lo))
        time.sleep(0.5)
    tri = riley_tri(np.array(elevs, dtype=float).reshape(GRID_N, GRID_N))
    filas.append({"planta": planta, "TRI_m": tri, "nMAE": nmae})
    print(f"  {planta:15s} TRI={tri:7.2f} m  nMAE={nmae:.3f}")

df_tri = pd.DataFrame(filas).sort_values("nMAE")
rho, p_s = stats.spearmanr(df_tri["TRI_m"], df_tri["nMAE"])
r, p_p = stats.pearsonr(df_tri["TRI_m"], df_tri["nMAE"])
print("\n" + df_tri.to_string(index=False))
print(f"\nSpearman rho={rho:.3f} p={p_s:.3f}  |  Pearson r={r:.3f} p={p_p:.3f}")
print("n=11 (df=9): r crítico a alpha=0.05 (dos colas) ~0.602")

df_tri.to_csv("tri_vs_nmae.csv", index=False)
fig, ax = plt.subplots(figsize=(3.4, 2.8))
ax.scatter(df_tri["TRI_m"], df_tri["nMAE"]*100, s=28, zorder=3)
for _, row in df_tri.iterrows():
    ax.annotate(row["planta"], (row["TRI_m"], row["nMAE"]*100), fontsize=6, xytext=(3,2), textcoords="offset points")
if p_p < 0.10:
    m, b = np.polyfit(df_tri["TRI_m"], df_tri["nMAE"]*100, 1)
    xs = np.linspace(df_tri["TRI_m"].min(), df_tri["TRI_m"].max(), 50)
    ax.plot(xs, m*xs+b, "--", lw=0.8, color="gray", zorder=2)
ax.set_xlabel("TRI (m)", fontsize=8); ax.set_ylabel("nMAE (%)", fontsize=8)
ax.tick_params(labelsize=7); fig.tight_layout()
fig.savefig("reports/tri_vs_nmae.pdf", bbox_inches="tight")
print("\nEscrito: tri_vs_nmae.csv, reports/tri_vs_nmae.pdf")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from pathlib import Path

PRED_DIR = Path("models/walkforward_v1/preds")
QUANTILES = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
MODELOS_OBS26 = ["WeightedEnsemble", "DirectTabular", "Chronos2"]

# concatenar todos los dias
df_all = pd.concat([pd.read_parquet(f) for f in sorted(PRED_DIR.glob("*.parquet"))], ignore_index=True)
df_all = df_all[df_all["model"].isin(MODELOS_OBS26)].sort_values(["item_id","model","timestamp"])

def sql_por_planta_modelo(g):
    yt = g["potencia_mw"].values
    mask = ~np.isnan(yt)
    if mask.sum() == 0:
        return np.nan
    yt_ = yt[mask]
    pinballs = [np.mean(np.where(yt_ >= g[str(q)].values[mask],
                                   q*(yt_ - g[str(q)].values[mask]),
                                   (1-q)*(g[str(q)].values[mask] - yt_)))
                for q in QUANTILES]
    y_naive = g["potencia_mw"].shift(24).values[mask]
    mask2 = ~np.isnan(y_naive)
    if mask2.sum() == 0:
        return np.nan
    return 2*np.mean(pinballs) / (np.mean(np.abs(yt_[mask2] - y_naive[mask2])) + 1e-8)

sql_tabla = (df_all.groupby(["item_id","model"])
             .apply(sql_por_planta_modelo, include_groups=False)
             .unstack("model")[MODELOS_OBS26])

print(sql_tabla.round(4))
print()

# Wilcoxon pareado, WeightedEnsemble vs cada uno de los otros dos
for otro in ["DirectTabular", "Chronos2"]:
    valid = sql_tabla[["WeightedEnsemble", otro]].dropna()
    stat, p = wilcoxon(valid["WeightedEnsemble"], valid[otro])
    print(f"WeightedEnsemble vs {otro}: n={len(valid)}  W={stat:.2f}  p={p:.4f}")

In [ ]:
import pickle
import numpy as np
from scipy.stats import wilcoxon
from pathlib import Path

df_res3 = pickle.load(open("models/tres_condiciones_v1/nmae_3cond.pkl", "rb"))
tabla22 = df_res3.pivot(index="item_id", columns="fuente", values="nMAE")[["era5","om_crudo","om_corr"]]
print(tabla22.round(4))
print()

for otro in ["om_crudo", "om_corr"]:
    valid = tabla22[["era5", otro]].dropna()
    stat, p = wilcoxon(valid["era5"], valid[otro])
    print(f"ERA5-Land vs {otro}: n={len(valid)}  W={stat:.2f}  p={p:.4f}")